# Agente de Trading Cuantitativo — SOLUSDT 5m

Notebook reorganizado para continuar el proyecto en Codex.

**Estado actual:** V2 multiclase (`0 = NO OPERAR`, `1 = LONG`, `2 = SHORT`). El OOS queda reservado para la evaluación final y no debe utilizarse para seleccionar filtros o parámetros.

## Orden de trabajo
1. Preparación de datos e indicadores.
2. Investigación que llevó al target V2.
3. Construcción del target V2 definitivo.
4. Features y split temporal purgado.
5. Entrenamiento Random Forest V2.
6. Diagnóstico exclusivamente sobre Validation.
7. Backtest auditado de Validation.
8. Auditoría de operaciones que alcanzan 1R.
9. Próximo desarrollo: filtro de calidad de señales, sin tocar OOS.

> Las versiones V1, Agente V3 y pruebas OOS anteriores se conservan al final como archivo histórico y **no forman parte del pipeline V2 activo**.


## 1. Preparación de datos e indicadores

Este bloque conserva la preparación necesaria heredada del notebook original, pero se detiene antes del simulador/meta-modelo antiguo. El CSV se espera en la carpeta de trabajo con el nombre `SOLUSDT_5m_futures.csv`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import RandomForestClassifier


# ==========================================================
# CONFIGURACIÓN
# ==========================================================

DATA_FILE = 'SOLUSDT_5m_futures.csv'
MODEL_FILE = 'meta_model.joblib'

INITIAL_CAPITAL = 10_000.0

# 35% del capital como margen/exposición base
POSITION_FRACTION = 0.35

# Apalancamiento utilizado por la simulación
LEVERAGE = 2.0

# Comisión + slippage por operación completa
ROUND_COST = (0.0004 + 0.0002) * 2

# División temporal:
# 56% TRAIN
# 14% VALIDATION
# 30% TEST OOS
TRAIN_PCT = 0.56
VALIDATION_PCT = 0.70


# ==========================================================
# 1. CARGA DE DATOS
# ==========================================================

print("1. Cargando datos y construyendo indicadores...")

df = pd.read_csv(
    DATA_FILE,
    index_col='timestamp',
    parse_dates=True
)

df.sort_index(inplace=True)

required_cols = [
    'open',
    'high',
    'low',
    'close',
    'volume'
]

missing = [
    c for c in required_cols
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Faltan columnas obligatorias: {missing}"
    )

close = df['close'].values
high = df['high'].values
low = df['low'].values
open_p = df['open'].values
vol = df['volume'].values

n_bars = len(df)


# ==========================================================
# 2. OPEN INTEREST
# ==========================================================

oi_candidates = [
    'open_interest',
    'openInterest',
    'oi',
    'OI',
    'Open Interest',
    'open_interest_value'
]

oi_col = next(
    (
        c for c in oi_candidates
        if c in df.columns
    ),
    None
)

if oi_col is not None:

    oi = pd.to_numeric(
        df[oi_col],
        errors='coerce'
    ).ffill().fillna(0.0)

    # Cambio porcentual de OI.
    # Se utilizará desplazado una vela en las entradas.
    oi_change = (
        oi.pct_change()
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .values
    )

    print(
        f"Open Interest detectado: {oi_col}"
    )

else:

    print(
        "⚠️ No se encontró Open Interest "
        "en el CSV. Se utilizará 0.0."
    )

    oi_change = np.zeros(n_bars)


# ==========================================================
# 3. INDICADORES MULTITEMPORALES
# ==========================================================

# ----------------------------------------------------------
# MACRO 1H
# ----------------------------------------------------------

df_1h = (
    df.resample('1h')
    .agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    })
    .dropna()
)

h1_ema50 = (
    df_1h['close']
    .ewm(span=50, adjust=False)
    .mean()
)

h1_ema200 = (
    df_1h['close']
    .ewm(span=200, adjust=False)
    .mean()
)

macro_bullish = (
    (
        (df_1h['close'] > h1_ema200) &
        (h1_ema50 > h1_ema200)
    )
    .shift(1)
    .reindex(df.index, method='ffill')
    .fillna(False)
    .values
)

macro_bearish = (
    (
        (df_1h['close'] < h1_ema200) &
        (h1_ema50 < h1_ema200)
    )
    .shift(1)
    .reindex(df.index, method='ffill')
    .fillna(False)
    .values
)


# ----------------------------------------------------------
# TENDENCIA 15M
# ----------------------------------------------------------

df_15m = (
    df.resample('15min')
    .agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    })
    .dropna()
)

m15_ema50 = (
    df_15m['close']
    .ewm(span=50, adjust=False)
    .mean()
)

trend_15m_bull = (
    (df_15m['close'] >= m15_ema50)
    .shift(1)
    .reindex(df.index, method='ffill')
    .fillna(False)
    .values
)

trend_15m_bear = (
    (df_15m['close'] <= m15_ema50)
    .shift(1)
    .reindex(df.index, method='ffill')
    .fillna(False)
    .values
)


# ==========================================================
# 4. INDICADORES 5M
# ==========================================================

ema20 = (
    df['close']
    .ewm(span=20, adjust=False)
    .mean()
    .values
)

ema50 = (
    df['close']
    .ewm(span=50, adjust=False)
    .mean()
    .values
)


# ----------------------------------------------------------
# BOLLINGER
# ----------------------------------------------------------

bb_mid = (
    df['close']
    .rolling(20)
    .mean()
    .values
)

bb_std = (
    df['close']
    .rolling(20)
    .std()
    .values
)

bb_up = bb_mid + 2 * bb_std
bb_low = bb_mid - 2 * bb_std

bb_den = bb_up - bb_low

bb_pct_b = np.full(
    n_bars,
    0.5,
    dtype=float
)

bb_mask = (
    np.isfinite(bb_den) &
    (bb_den != 0)
)

bb_pct_b[bb_mask] = (
    (close[bb_mask] - bb_low[bb_mask])
    / bb_den[bb_mask]
)


# ----------------------------------------------------------
# ATR
# ----------------------------------------------------------

prev_close = np.roll(close, 1)

tr = np.maximum(
    high - low,
    np.maximum(
        np.abs(high - prev_close),
        np.abs(low - prev_close)
    )
)

tr[0] = high[0] - low[0]

atr = (
    pd.Series(
        tr,
        index=df.index
    )
    .rolling(14)
    .mean()
    .values
)


# ----------------------------------------------------------
# ADX
# ----------------------------------------------------------

prev_high = np.roll(high, 1)
prev_low = np.roll(low, 1)

up_m = high - prev_high
down_m = prev_low - low

p_dm = np.where(
    (up_m > down_m) & (up_m > 0),
    up_m,
    0.0
)

m_dm = np.where(
    (down_m > up_m) & (down_m > 0),
    down_m,
    0.0
)

safe_atr = np.where(
    np.isfinite(atr) & (atr > 0),
    atr,
    1.0
)

p_di = (
    100 *
    pd.Series(
        p_dm,
        index=df.index
    )
    .rolling(14)
    .mean()
    .values
    / safe_atr
)

m_di = (
    100 *
    pd.Series(
        m_dm,
        index=df.index
    )
    .rolling(14)
    .mean()
    .values
    / safe_atr
)

di_sum = np.where(
    np.isfinite(p_di + m_di) &
    ((p_di + m_di) > 0),
    p_di + m_di,
    1.0
)

adx = (
    pd.Series(
        100 * np.abs(p_di - m_di) / di_sum,
        index=df.index
    )
    .rolling(14)
    .mean()
    .fillna(20.0)
    .values
)


# ----------------------------------------------------------
# VOLUMEN
# ----------------------------------------------------------

vol_sma = (
    pd.Series(
        vol,
        index=df.index
    )
    .rolling(20)
    .mean()
    .values
)

vol_ratio = np.where(
    np.isfinite(vol_sma) & (vol_sma > 0),
    vol / vol_sma,
    1.0
)


# ==========================================================
# 5. RSI + STOCHRSI
# ==========================================================

delta = df['close'].diff()

gain = (
    delta
    .clip(lower=0)
    .rolling(14)
    .mean()
)

loss = (
    -delta
    .clip(upper=0)
    .rolling(14)
    .mean()
)

loss = loss.replace(0, np.nan)

rs = gain / loss

rsi_s = (
    100 - (100 / (1 + rs))
)

rsi_s = (
    rsi_s
    .replace([np.inf, -np.inf], np.nan)
    .fillna(50.0)
    .values
)

rsi_min = (
    pd.Series(
        rsi_s,
        index=df.index
    )
    .rolling(14)
    .min()
    .values
)

rsi_max = (
    pd.Series(
        rsi_s,
        index=df.index
    )
    .rolling(14)
    .max()
    .values
)

# ----------------------------------------------------------
# CORRECCIÓN DEL WARNING DEL STOCHRSI
# ----------------------------------------------------------

denom = rsi_max - rsi_min

stoch_raw = np.full(
    n_bars,
    50.0,
    dtype=float
)

stoch_mask = (
    np.isfinite(denom) &
    np.isfinite(rsi_s) &
    (denom != 0)
)

stoch_raw[stoch_mask] = (
    (
        rsi_s[stoch_mask]
        - rsi_min[stoch_mask]
    )
    / denom[stoch_mask]
) * 100

stoch_k = (
    pd.Series(
        stoch_raw,
        index=df.index
    )
    .rolling(3)
    .mean()
    .values
)

stoch_d = (
    pd.Series(
        stoch_k,
        index=df.index
    )
    .rolling(3)
    .mean()
    .values
)


# ==========================================================
# 6. SEÑALES
# ==========================================================

prev_k = np.roll(stoch_k, 1)
prev_d = np.roll(stoch_d, 1)

stoch_cross_long = (
    (prev_k < 30) &
    (stoch_k > stoch_d) &
    (prev_k <= prev_d)
)

stoch_cross_short = (
    (prev_k > 70) &
    (stoch_k < stoch_d) &
    (prev_k >= prev_d)
)


bullish_candle = (
    (close >= open_p) |
    ((close - low) > (high - close))
)

bearish_candle = (
    (close <= open_p) |
    ((high - close) > (close - low))
)


pullback_support = (
    (low <= ema20 * 1.002) |
    (bb_pct_b <= 0.38)
)

pullback_resistance = (
    (high >= ema20 * 0.998) |
    (bb_pct_b >= 0.62)
)


weekdays = df.index.weekday.values
hours = df.index.hour.values

is_weekend = (
    ((weekdays == 4) & (hours >= 21)) |
    (weekdays == 5) |
    (weekdays == 6)
)


# ----------------------------------------------------------
# FILTRO ADX
# ----------------------------------------------------------
#
# No imponemos un valor arbitrario.
# El ML será quien determine parte de la selección.
# ----------------------------------------------------------

sig_long = (
    macro_bullish &
    trend_15m_bull &
    stoch_cross_long &
    pullback_support &
    bullish_candle &
    (~is_weekend)
)

sig_short = (
    macro_bearish &
    trend_15m_bear &
    stoch_cross_short &
    pullback_resistance &
    bearish_candle &
    (~is_weekend)
)


# La señal se genera en la vela anterior.
# Entrada en OPEN de la vela siguiente.

entry_long = np.roll(sig_long, 1)
entry_long[0] = False

entry_short = np.roll(sig_short, 1)
entry_short[0] = False


# ==========================================================
# 7. FEATURES DEL META-MODELO
# ==========================================================

feature_cols = [
    'rsi_5m',
    'adx_5m',
    'atr_pct_5m',
    'volume_ratio_5m',
    'oi_change_5m',
    'hour_utc',
    'day_of_week',
    'side'
]


def get_features(i, is_long):
    """
    IMPORTANTE:
    La entrada ocurre en OPEN[i].

    Por eso TODAS las features utilizan i-1.
    No usamos información de cierre de la vela de entrada.
    """

    j = i - 1

    if j < 0:
        return None

    atr_value = atr[j]

    atr_pct = (
        atr_value / close[j]
        if np.isfinite(atr_value)
        and close[j] > 0
        else 0.0
    )

    feats = {
        'rsi_5m': float(rsi_s[j]),
        'adx_5m': float(adx[j]),
        'atr_pct_5m': float(atr_pct),
        'volume_ratio_5m': float(vol_ratio[j]),
        'oi_change_5m': float(oi_change[j]),
        'hour_utc': int(hours[j]),
        'day_of_week': int(weekdays[j]),
        'side': 1 if is_long else -1
    }

    # Protección NaN / Inf
    for k in feats:
        if not np.isfinite(feats[k]):
            feats[k] = 0.0

    return feats


# ==========================================================


## 2. Investigación del target V2

Experimentos de diseño del target. Se mantienen porque documentan por qué se eligió **1R dentro de 12 velas (1 hora)**.


In [ ]:
# ================================================================
# ANÁLISIS V2 OPTIMIZADO - TASAS DE ÉXITO POR MÚLTIPLO DE R
# ================================================================

import numpy as np
import pandas as pd

print("=" * 75)
print("ANÁLISIS V2 OPTIMIZADO - TARGETS BASADOS EN R")
print("=" * 75)

HORIZON_V2 = 3

ATR_MULTIPLIER = 1.2
R_MIN_PCT = 0.01
R_MAX_PCT = 0.025

TARGET_MULTIPLES = [0.5, 1.0, 1.5, 2.0, 2.5]

# ------------------------------------------------
# 1. ARRAYS BASE
# ------------------------------------------------

entry = df["open"].shift(-1).to_numpy(dtype=float)
atr = df["atr_14"].to_numpy(dtype=float)

# R dinámico:
# 1.2 * ATR, limitado entre 1% y 2.5% del precio de entrada

atr_distance = ATR_MULTIPLIER * atr
min_distance = R_MIN_PCT * entry
max_distance = R_MAX_PCT * entry

R = np.clip(
    atr_distance,
    min_distance,
    max_distance
)

# ------------------------------------------------
# 2. FUTUROS: próximas 3 velas
# ------------------------------------------------

future_highs = np.column_stack([
    df["high"].shift(-k).to_numpy(dtype=float)
    for k in range(1, HORIZON_V2 + 1)
])

future_lows = np.column_stack([
    df["low"].shift(-k).to_numpy(dtype=float)
    for k in range(1, HORIZON_V2 + 1)
])

# Solo filas realmente evaluables
valid = (
    np.isfinite(entry) &
    np.isfinite(R) &
    np.isfinite(future_highs).all(axis=1) &
    np.isfinite(future_lows).all(axis=1)
)

print(f"\nFilas totales: {len(df):,}")
print(f"Filas válidas para evaluar: {valid.sum():,}")
print(f"Filas descartadas: {(~valid).sum():,}")

# ------------------------------------------------
# 3. FUNCIÓN VECTORIZADA
# ------------------------------------------------

def evaluar_multiple_R(multiple):

    # ============================
    # LONG
    # ============================

    long_tp = entry + multiple * R
    long_sl = entry - R

    long_success = np.zeros(len(df), dtype=bool)
    long_failure = np.zeros(len(df), dtype=bool)

    unresolved = valid.copy()

    for k in range(HORIZON_V2):

        high = future_highs[:, k]
        low = future_lows[:, k]

        # Ambos niveles tocados en la misma vela.
        # Convención conservadora: SL primero.
        both = (
            unresolved &
            (high >= long_tp) &
            (low <= long_sl)
        )

        tp_only = (
            unresolved &
            (high >= long_tp) &
            (low > long_sl)
        )

        sl_only = (
            unresolved &
            (low <= long_sl) &
            (high < long_tp)
        )

        long_failure |= both | sl_only
        long_success |= tp_only

        # Ya ocurrió un evento, no seguimos mirando esa operación
        unresolved &= ~(both | tp_only | sl_only)

    # ============================
    # SHORT
    # ============================

    short_tp = entry - multiple * R
    short_sl = entry + R

    short_success = np.zeros(len(df), dtype=bool)
    short_failure = np.zeros(len(df), dtype=bool)

    unresolved = valid.copy()

    for k in range(HORIZON_V2):

        high = future_highs[:, k]
        low = future_lows[:, k]

        # Ambos niveles tocados en la misma vela.
        # Convención conservadora: SL primero.
        both = (
            unresolved &
            (low <= short_tp) &
            (high >= short_sl)
        )

        tp_only = (
            unresolved &
            (low <= short_tp) &
            (high < short_sl)
        )

        sl_only = (
            unresolved &
            (high >= short_sl) &
            (low > short_tp)
        )

        short_failure |= both | sl_only
        short_success |= tp_only

        unresolved &= ~(both | tp_only | sl_only)

    # ------------------------------------------------
    # 4. CLASIFICACIÓN
    # ------------------------------------------------

    # 1 = LONG
    # 2 = SHORT
    # 0 = NO OPERAR
    #
    # Si ambos lados consiguen TP dentro del horizonte,
    # lo consideramos ambiguo y queda como NO OPERAR.

    long_only = long_success & ~short_success
    short_only = short_success & ~long_success
    both_success = long_success & short_success

    no_trade = valid & ~(long_only | short_only)

    total_valid = valid.sum()

    return {
        "multiple": multiple,
        "no_trade": no_trade.sum(),
        "long": long_only.sum(),
        "short": short_only.sum(),
        "both_ambiguous": both_success.sum(),
        "total_valid": total_valid
    }


# ------------------------------------------------
# 5. EVALUAR TODOS LOS MÚLTIPLOS
# ------------------------------------------------

results = []

for multiple in TARGET_MULTIPLES:

    result = evaluar_multiple_R(multiple)
    results.append(result)

    total = result["total_valid"]

    print(f"\n{'=' * 60}")
    print(f"TARGET = {multiple}R")
    print(f"{'=' * 60}")

    print(
        f"NO OPERAR:       {result['no_trade']:>8,} "
        f"({result['no_trade']/total:.2%})"
    )

    print(
        f"LONG:            {result['long']:>8,} "
        f"({result['long']/total:.2%})"
    )

    print(
        f"SHORT:           {result['short']:>8,} "
        f"({result['short']/total:.2%})"
    )

    print(
        f"AMBIGUO:         {result['both_ambiguous']:>8,} "
        f"({result['both_ambiguous']/total:.2%})"
    )

    print(
        f"TOTAL OPERABLE:  "
        f"{(result['long'] + result['short']):>8,} "
        f"({(result['long'] + result['short'])/total:.2%})"
    )


# ------------------------------------------------
# 6. TABLA RESUMEN
# ------------------------------------------------

summary = pd.DataFrame(results)

summary["long_pct"] = summary["long"] / summary["total_valid"]
summary["short_pct"] = summary["short"] / summary["total_valid"]
summary["operable_pct"] = (
    (summary["long"] + summary["short"])
    / summary["total_valid"]
)

print("\n")
print("=" * 75)
print("RESUMEN")
print("=" * 75)

print(
    summary[
        [
            "multiple",
            "long",
            "short",
            "no_trade",
            "both_ambiguous",
            "long_pct",
            "short_pct",
            "operable_pct"
        ]
    ].to_string(index=False)
)

print("\nAnálisis terminado.")



In [ ]:
# ================================================================
# ANÁLISIS V2 - 1R VS DISTINTOS HORIZONTES
# ================================================================

import numpy as np
import pandas as pd

print("=" * 75)
print("ANÁLISIS 1R - DISTINTOS HORIZONTES")
print("=" * 75)

ATR_MULTIPLIER = 1.2
R_MIN_PCT = 0.01
R_MAX_PCT = 0.025

TARGET_R = 1.0

HORIZONS = [3, 6, 12, 24, 36]

# ------------------------------------------------
# 1. PRECIO DE ENTRADA Y R
# ------------------------------------------------

entry = df["open"].shift(-1).to_numpy(dtype=float)
atr = df["atr_14"].to_numpy(dtype=float)

atr_distance = ATR_MULTIPLIER * atr
min_distance = R_MIN_PCT * entry
max_distance = R_MAX_PCT * entry

R = np.clip(
    atr_distance,
    min_distance,
    max_distance
)

# ------------------------------------------------
# 2. FUNCIÓN DE EVALUACIÓN
# ------------------------------------------------

def evaluar_horizonte(horizon):

    future_highs = np.column_stack([
        df["high"].shift(-k).to_numpy(dtype=float)
        for k in range(1, horizon + 1)
    ])

    future_lows = np.column_stack([
        df["low"].shift(-k).to_numpy(dtype=float)
        for k in range(1, horizon + 1)
    ])

    valid = (
        np.isfinite(entry) &
        np.isfinite(R) &
        np.isfinite(future_highs).all(axis=1) &
        np.isfinite(future_lows).all(axis=1)
    )

    # ============================================================
    # LONG
    # ============================================================

    long_tp = entry + TARGET_R * R
    long_sl = entry - R

    long_success = np.zeros(len(df), dtype=bool)
    long_failure = np.zeros(len(df), dtype=bool)

    unresolved = valid.copy()

    for k in range(horizon):

        high = future_highs[:, k]
        low = future_lows[:, k]

        both = (
            unresolved &
            (high >= long_tp) &
            (low <= long_sl)
        )

        tp_only = (
            unresolved &
            (high >= long_tp) &
            (low > long_sl)
        )

        sl_only = (
            unresolved &
            (low <= long_sl) &
            (high < long_tp)
        )

        long_failure |= both | sl_only
        long_success |= tp_only

        unresolved &= ~(both | tp_only | sl_only)

    # ============================================================
    # SHORT
    # ============================================================

    short_tp = entry - TARGET_R * R
    short_sl = entry + R

    short_success = np.zeros(len(df), dtype=bool)
    short_failure = np.zeros(len(df), dtype=bool)

    unresolved = valid.copy()

    for k in range(horizon):

        high = future_highs[:, k]
        low = future_lows[:, k]

        both = (
            unresolved &
            (low <= short_tp) &
            (high >= short_sl)
        )

        tp_only = (
            unresolved &
            (low <= short_tp) &
            (high < short_sl)
        )

        sl_only = (
            unresolved &
            (high >= short_sl) &
            (low > short_tp)
        )

        short_failure |= both | sl_only
        short_success |= tp_only

        unresolved &= ~(both | tp_only | sl_only)

    # ============================================================
    # CLASIFICACIÓN FINAL
    # ============================================================

    long_only = long_success & ~short_success
    short_only = short_success & ~long_success
    ambiguous = long_success & short_success

    no_trade = valid & ~(long_only | short_only)

    total = valid.sum()

    return {
        "horizon": horizon,
        "minutes": horizon * 5,
        "valid": total,
        "long": long_only.sum(),
        "short": short_only.sum(),
        "ambiguous": ambiguous.sum(),
        "no_trade": no_trade.sum(),
        "operable": long_only.sum() + short_only.sum(),
        "operable_pct": (
            (long_only.sum() + short_only.sum()) / total
        )
    }


# ------------------------------------------------
# 3. EJECUTAR
# ------------------------------------------------

results = []

for horizon in HORIZONS:

    result = evaluar_horizonte(horizon)
    results.append(result)

    print("\n" + "=" * 60)
    print(
        f"HORIZONTE: {horizon} velas "
        f"({horizon * 5} minutos)"
    )
    print("=" * 60)

    print(
        f"LONG:             {result['long']:>8,} "
        f"({result['long']/result['valid']:.2%})"
    )

    print(
        f"SHORT:            {result['short']:>8,} "
        f"({result['short']/result['valid']:.2%})"
    )

    print(
        f"NO OPERAR:        {result['no_trade']:>8,} "
        f"({result['no_trade']/result['valid']:.2%})"
    )

    print(
        f"AMBIGUO:          {result['ambiguous']:>8,} "
        f"({result['ambiguous']/result['valid']:.2%})"
    )

    print(
        f"TOTAL OPERABLE:   {result['operable']:>8,} "
        f"({result['operable_pct']:.2%})"
    )


# ------------------------------------------------
# 4. RESUMEN
# ------------------------------------------------

summary_1R = pd.DataFrame(results)

print("\n")
print("=" * 75)
print("RESUMEN 1R VS HORIZONTE")
print("=" * 75)

print(
    summary_1R[
        [
            "horizon",
            "minutes",
            "long",
            "short",
            "no_trade",
            "ambiguous",
            "operable",
            "operable_pct"
        ]
    ].to_string(index=False)
)

print("\nAnálisis terminado.")

In [ ]:
# ================================================================
# ANÁLISIS V2 - ESCALERA DE R A 1 HORA
# ================================================================

import numpy as np
import pandas as pd

print("=" * 75)
print("ANÁLISIS DE ESCALERA R - HORIZONTE 1 HORA")
print("=" * 75)

HORIZON = 12

ATR_MULTIPLIER = 1.2
R_MIN_PCT = 0.01
R_MAX_PCT = 0.025

R_MULTIPLES = [1.0, 1.5, 2.5, 3.8]

# ------------------------------------------------
# 1. PRECIO DE ENTRADA Y R
# ------------------------------------------------

entry = df["open"].shift(-1).to_numpy(dtype=float)
atr = df["atr_14"].to_numpy(dtype=float)

atr_distance = ATR_MULTIPLIER * atr
min_distance = R_MIN_PCT * entry
max_distance = R_MAX_PCT * entry

R = np.clip(
    atr_distance,
    min_distance,
    max_distance
)

# ------------------------------------------------
# 2. FUTURAS 12 VELAS
# ------------------------------------------------

future_highs = np.column_stack([
    df["high"].shift(-k).to_numpy(dtype=float)
    for k in range(1, HORIZON + 1)
])

future_lows = np.column_stack([
    df["low"].shift(-k).to_numpy(dtype=float)
    for k in range(1, HORIZON + 1)
])

valid = (
    np.isfinite(entry) &
    np.isfinite(R) &
    np.isfinite(future_highs).all(axis=1) &
    np.isfinite(future_lows).all(axis=1)
)

total_valid = valid.sum()

print(f"\nFilas totales: {len(df):,}")
print(f"Filas válidas: {total_valid:,}")

# ================================================================
# FUNCIÓN PARA ANALIZAR UNA DIRECCIÓN
# ================================================================

def analizar_direccion(direction, multiple):

    if direction == "LONG":

        tp = entry + multiple * R
        sl = entry - R

    else:

        tp = entry - multiple * R
        sl = entry + R

    success = np.zeros(len(df), dtype=bool)
    failure = np.zeros(len(df), dtype=bool)

    unresolved = valid.copy()

    for k in range(HORIZON):

        high = future_highs[:, k]
        low = future_lows[:, k]

        if direction == "LONG":

            both = (
                unresolved &
                (high >= tp) &
                (low <= sl)
            )

            tp_only = (
                unresolved &
                (high >= tp) &
                (low > sl)
            )

            sl_only = (
                unresolved &
                (low <= sl) &
                (high < tp)
            )

        else:

            both = (
                unresolved &
                (low <= tp) &
                (high >= sl)
            )

            tp_only = (
                unresolved &
                (low <= tp) &
                (high < sl)
            )

            sl_only = (
                unresolved &
                (high >= sl) &
                (low > tp)
            )

        # Si toca TP y SL en la misma vela:
        # convención conservadora = SL primero

        success |= tp_only
        failure |= both | sl_only

        unresolved &= ~(both | tp_only | sl_only)

    unresolved &= valid

    return {
        "success": success.sum(),
        "failure": failure.sum(),
        "unresolved": unresolved.sum()
    }


# ================================================================
# 3. ANALIZAR LONG Y SHORT
# ================================================================

results = []

for multiple in R_MULTIPLES:

    long_result = analizar_direccion("LONG", multiple)
    short_result = analizar_direccion("SHORT", multiple)

    long_success_pct = (
        long_result["success"] / total_valid
    )

    short_success_pct = (
        short_result["success"] / total_valid
    )

    results.append({
        "R": multiple,

        "LONG_exito": long_result["success"],
        "LONG_exito_pct": long_success_pct,

        "LONG_SL": long_result["failure"],

        "LONG_sin_evento": long_result["unresolved"],

        "SHORT_exito": short_result["success"],
        "SHORT_exito_pct": short_success_pct,

        "SHORT_SL": short_result["failure"],

        "SHORT_sin_evento": short_result["unresolved"]
    })

# ================================================================
# 4. MOSTRAR RESULTADOS
# ================================================================

summary_R = pd.DataFrame(results)

for _, row in summary_R.iterrows():

    print("\n" + "=" * 65)
    print(f"TARGET = {row['R']}R")
    print("=" * 65)

    print(
        f"LONG  → TP: {int(row['LONG_exito']):>8,} "
        f"({row['LONG_exito_pct']:.2%})"
    )

    print(
        f"LONG  → SL: {int(row['LONG_SL']):>8,}"
    )

    print(
        f"LONG  → sin evento: "
        f"{int(row['LONG_sin_evento']):>8,}"
    )

    print()

    print(
        f"SHORT → TP: {int(row['SHORT_exito']):>8,} "
        f"({row['SHORT_exito_pct']:.2%})"
    )

    print(
        f"SHORT → SL: {int(row['SHORT_SL']):>8,}"
    )

    print(
        f"SHORT → sin evento: "
        f"{int(row['SHORT_sin_evento']):>8,}"
    )


# ================================================================
# 5. TABLA FINAL
# ================================================================

print("\n")
print("=" * 75)
print("RESUMEN ESCALERA DE R")
print("=" * 75)

print(
    summary_R[
        [
            "R",
            "LONG_exito_pct",
            "SHORT_exito_pct",
            "LONG_exito",
            "SHORT_exito"
        ]
    ].to_string(index=False)
)

print("\nAnálisis terminado.")

## 3. Target V2 definitivo

Target multiclase basado en alcanzar 1R antes del stop dentro de las siguientes 12 velas.


In [ ]:
# ================================================================
# V2 - CONSTRUCCIÓN DEL TARGET DEFINITIVO
# ================================================================

import numpy as np
import pandas as pd

print("=" * 75)
print("CONSTRUCCIÓN TARGET V2")
print("=" * 75)

# ------------------------------------------------
# CONFIGURACIÓN
# ------------------------------------------------

HORIZON_V2 = 12          # 12 velas de 5 min = 1 hora
TARGET_R = 1.0

ATR_MULTIPLIER = 1.2
R_MIN_PCT = 0.01
R_MAX_PCT = 0.025

# ------------------------------------------------
# 1. PRECIO DE ENTRADA
# ------------------------------------------------

entry = df["open"].shift(-1).to_numpy(dtype=float)

# ATR de la vela de señal
atr = df["atr_14"].to_numpy(dtype=float)

# ------------------------------------------------
# 2. CALCULAR R
# ------------------------------------------------

atr_distance = ATR_MULTIPLIER * atr
min_distance = R_MIN_PCT * entry
max_distance = R_MAX_PCT * entry

R = np.clip(
    atr_distance,
    min_distance,
    max_distance
)

# ------------------------------------------------
# 3. NIVELES LONG / SHORT
# ------------------------------------------------

long_tp = entry + TARGET_R * R
long_sl = entry - R

short_tp = entry - TARGET_R * R
short_sl = entry + R

# ------------------------------------------------
# 4. FUTURAS 12 VELAS
# ------------------------------------------------

future_highs = np.column_stack([
    df["high"].shift(-k).to_numpy(dtype=float)
    for k in range(1, HORIZON_V2 + 1)
])

future_lows = np.column_stack([
    df["low"].shift(-k).to_numpy(dtype=float)
    for k in range(1, HORIZON_V2 + 1)
])

# ------------------------------------------------
# 5. FILAS VÁLIDAS
# ------------------------------------------------

valid = (
    np.isfinite(entry) &
    np.isfinite(R) &
    np.isfinite(future_highs).all(axis=1) &
    np.isfinite(future_lows).all(axis=1)
)

print(f"\nFilas totales: {len(df):,}")
print(f"Filas válidas: {valid.sum():,}")
print(f"Filas inválidas: {(~valid).sum():,}")

# ------------------------------------------------
# 6. TARGET INICIAL
# ------------------------------------------------

# NaN = fila que no puede utilizarse
#
# 0 = NO OPERAR
# 1 = LONG
# 2 = SHORT

target_v2 = np.full(len(df), np.nan)

target_v2[valid] = 0

# Estados independientes
long_success = np.zeros(len(df), dtype=bool)
short_success = np.zeros(len(df), dtype=bool)

long_failure = np.zeros(len(df), dtype=bool)
short_failure = np.zeros(len(df), dtype=bool)

# ------------------------------------------------
# 7. EVALUAR VELA POR VELA
# ------------------------------------------------

long_unresolved = valid.copy()
short_unresolved = valid.copy()

for k in range(HORIZON_V2):

    high = future_highs[:, k]
    low = future_lows[:, k]

    # ============================================================
    # LONG
    # ============================================================

    long_both = (
        long_unresolved &
        (high >= long_tp) &
        (low <= long_sl)
    )

    long_tp_only = (
        long_unresolved &
        (high >= long_tp) &
        (low > long_sl)
    )

    long_sl_only = (
        long_unresolved &
        (low <= long_sl) &
        (high < long_tp)
    )

    long_success |= long_tp_only
    long_failure |= long_both | long_sl_only

    long_unresolved &= ~(
        long_both |
        long_tp_only |
        long_sl_only
    )

    # ============================================================
    # SHORT
    # ============================================================

    short_both = (
        short_unresolved &
        (low <= short_tp) &
        (high >= short_sl)
    )

    short_tp_only = (
        short_unresolved &
        (low <= short_tp) &
        (high < short_sl)
    )

    short_sl_only = (
        short_unresolved &
        (high >= short_sl) &
        (low > short_tp)
    )

    short_success |= short_tp_only
    short_failure |= short_both | short_sl_only

    short_unresolved &= ~(
        short_both |
        short_tp_only |
        short_sl_only
    )

# ------------------------------------------------
# 8. ASIGNAR CLASES
# ------------------------------------------------

# Solo LONG
long_only = (
    long_success &
    ~short_success
)

# Solo SHORT
short_only = (
    short_success &
    ~long_success
)

# Ambos consiguen TP:
# caso ambiguo → NO OPERAR
both_success = (
    long_success &
    short_success
)

# Asignación
target_v2[long_only] = 1
target_v2[short_only] = 2

# Los demás válidos permanecen en 0
# (incluye sin evento y casos ambiguos)

# ------------------------------------------------
# 9. GUARDAR EN DF
# ------------------------------------------------

df["target_v2"] = target_v2

# ------------------------------------------------
# 10. RESUMEN
# ------------------------------------------------

valid_target = df["target_v2"].notna()

counts = df.loc[valid_target, "target_v2"].value_counts().sort_index()

total = counts.sum()

print("\n" + "=" * 75)
print("DISTRIBUCIÓN TARGET V2")
print("=" * 75)

print(
    f"NO OPERAR (0): {int(counts.get(0, 0)):>10,} "
    f"({counts.get(0, 0) / total:.2%})"
)

print(
    f"LONG (1):      {int(counts.get(1, 0)):>10,} "
    f"({counts.get(1, 0) / total:.2%})"
)

print(
    f"SHORT (2):     {int(counts.get(2, 0)):>10,} "
    f"({counts.get(2, 0) / total:.2%})"
)

print(
    f"TOTAL VÁLIDO:  {int(total):>10,}"
)

print(
    f"INVÁLIDO:      {int((~valid_target).sum()):>10,}"
)

print("\nTarget V2 creado correctamente.")

## 4. Features y split temporal purgado

El gap temporal evita contaminación por el horizonte futuro del target.


In [ ]:
# ================================================================
# V2 - FEATURES + SPLIT TEMPORAL
# ================================================================

import numpy as np
import pandas as pd

print("=" * 75)
print("V2 - PREPARACIÓN TRAIN / VALIDATION / OOS")
print("=" * 75)

# ------------------------------------------------
# 1. FEATURES
# ------------------------------------------------

FEATURES_V2 = [
    "return_1",
    "return_3",
    "return_6",
    "return_12",
    "return_24",
    "volatility_12",
    "volatility_24",
    "sma_10",
    "sma_20",
    "sma_50",
    "sma_100",
    "ema_9",
    "ema_20",
    "ema_50",
    "close_vs_ema20",
    "close_vs_ema50",
    "rsi_14",
    "macd",
    "macd_signal",
    "macd_hist",
    "atr_14",
    "atr_pct",
    "volume",
    "volume_sma_20",
    "volume_ratio",
    "high_low_range",
    "body_size",
    "upper_wick",
    "lower_wick"
]

print(f"\nCantidad de features: {len(FEATURES_V2)}")

# ------------------------------------------------
# 2. CREAR DATASET DE MODELADO
# ------------------------------------------------

model_df = df[
    ["timestamp", "target_v2"] + FEATURES_V2
].copy()

# Eliminar solamente filas con target inválido
model_df = model_df[
    model_df["target_v2"].notna()
].copy()

# Eliminar NaN de features
model_df = model_df.dropna(
    subset=FEATURES_V2
).copy()

# Target entero
model_df["target_v2"] = model_df["target_v2"].astype(int)

print(f"\nFilas utilizables para modelo: {len(model_df):,}")

# ------------------------------------------------
# 3. SPLIT TEMPORAL
# ------------------------------------------------

TRAIN_END = "2025-12-31 00:00:00"
VALIDATION_END = "2026-06-30 00:00:00"

train = model_df[
    model_df["timestamp"] <= TRAIN_END
].copy()

validation = model_df[
    (model_df["timestamp"] > TRAIN_END) &
    (model_df["timestamp"] <= VALIDATION_END)
].copy()

oos = model_df[
    model_df["timestamp"] > VALIDATION_END
].copy()

# ------------------------------------------------
# 4. X / y
# ------------------------------------------------

X_train = train[FEATURES_V2]
y_train = train["target_v2"]

X_validation = validation[FEATURES_V2]
y_validation = validation["target_v2"]

X_oos = oos[FEATURES_V2]
y_oos = oos["target_v2"]

# ------------------------------------------------
# 5. RESUMEN
# ------------------------------------------------

print("\n" + "=" * 75)
print("SPLIT TEMPORAL")
print("=" * 75)

print(
    f"TRAIN:       {train['timestamp'].min()} "
    f"→ {train['timestamp'].max()} "
    f"| {len(train):,} filas"
)

print(
    f"VALIDATION:  {validation['timestamp'].min()} "
    f"→ {validation['timestamp'].max()} "
    f"| {len(validation):,} filas"
)

print(
    f"OOS:         {oos['timestamp'].min()} "
    f"→ {oos['timestamp'].max()} "
    f"| {len(oos):,} filas"
)

# ------------------------------------------------
# 6. DISTRIBUCIÓN DE CLASES
# ------------------------------------------------

print("\n" + "=" * 75)
print("DISTRIBUCIÓN DE CLASES")
print("=" * 75)

for nombre, y in [
    ("TRAIN", y_train),
    ("VALIDATION", y_validation),
    ("OOS", y_oos)
]:

    counts = y.value_counts().sort_index()
    total = len(y)

    print(f"\n{nombre}")

    for clase, etiqueta in [
        (0, "NO OPERAR"),
        (1, "LONG"),
        (2, "SHORT")
    ]:

        cantidad = counts.get(clase, 0)

        print(
            f"  {clase} - {etiqueta:<10}: "
            f"{cantidad:>8,} "
            f"({cantidad / total:.2%})"
        )


        # ================================================================
# V2 — SPLIT TEMPORAL PURGADO
# ================================================================

print("=" * 75)
print("V2 — SPLIT TEMPORAL PURGADO")
print("=" * 75)

HORIZON_V2 = 12

# Asegurar timestamp como datetime
model_df["timestamp"] = pd.to_datetime(model_df["timestamp"])

# Cortes originales
TRAIN_END = pd.Timestamp("2025-12-31 00:00:00")
VALIDATION_END = pd.Timestamp("2026-06-30 00:00:00")

# Gap de 12 velas = 1 hora
PURGE = pd.Timedelta(minutes=5 * HORIZON_V2)

# ------------------------------------------------
# TRAIN
# Hasta TRAIN_END
# ------------------------------------------------
train = model_df[
    model_df["timestamp"] <= TRAIN_END
].copy()

# ------------------------------------------------
# VALIDATION
# Empieza después de TRAIN_END + 1 hora
# ------------------------------------------------
validation = model_df[
    (model_df["timestamp"] > TRAIN_END + PURGE) &
    (model_df["timestamp"] <= VALIDATION_END)
].copy()

# ------------------------------------------------
# OOS
# Empieza después de VALIDATION_END + 1 hora
# ------------------------------------------------
oos = model_df[
    model_df["timestamp"] > VALIDATION_END + PURGE
].copy()

# ------------------------------------------------
# Separar X / y
# ------------------------------------------------

X_train = train[FEATURES_V2].copy()
y_train = train["target_v2"].astype(int).copy()

X_validation = validation[FEATURES_V2].copy()
y_validation = validation["target_v2"].astype(int).copy()

X_oos = oos[FEATURES_V2].copy()
y_oos = oos["target_v2"].astype(int).copy()

# ================================================================
# RESUMEN
# ================================================================

print("\nFECHAS:")

print(
    f"TRAIN:       {train['timestamp'].min()} "
    f"→ {train['timestamp'].max()}"
)

print(
    f"VALIDATION:  {validation['timestamp'].min()} "
    f"→ {validation['timestamp'].max()}"
)

print(
    f"OOS:         {oos['timestamp'].min()} "
    f"→ {oos['timestamp'].max()}"
)

print("\nTAMAÑOS:")

print(f"TRAIN:       {len(train):,}")
print(f"VALIDATION:  {len(validation):,}")
print(f"OOS:         {len(oos):,}")

print("\nDISTRIBUCIÓN TRAIN:")
print(
    y_train.value_counts()
    .sort_index()
    .rename(index={
        0: "NO OPERAR",
        1: "LONG",
        2: "SHORT"
    })
)

print("\nDISTRIBUCIÓN VALIDATION:")
print(
    y_validation.value_counts()
    .sort_index()
    .rename(index={
        0: "NO OPERAR",
        1: "LONG",
        2: "SHORT"
    })
)

print("\nDISTRIBUCIÓN OOS:")
print(
    y_oos.value_counts()
    .sort_index()
    .rename(index={
        0: "NO OPERAR",
        1: "LONG",
        2: "SHORT"
    })
)

print("\nGAPS:")

print(
    "TRAIN → VALIDATION:",
    validation["timestamp"].min() - train["timestamp"].max()
)

print(
    "VALIDATION → OOS:",
    oos["timestamp"].min() - validation["timestamp"].max()
)

print("\n✓ Split temporal purgado creado.")

## 5. Entrenamiento Random Forest V2

Entrenamiento y evaluación del modelo multiclase. El OOS puede calcularse aquí para auditoría histórica del notebook, pero **no debe usarse para elegir nuevas reglas, filtros o thresholds**.


In [ ]:
# ================================================================
# ENTRENAMIENTO RANDOM FOREST V2 — SPLIT PURGADO
# ================================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("=" * 75)
print("ENTRENAMIENTO RANDOM FOREST V2 — SPLIT PURGADO")
print("=" * 75)

# ------------------------------------------------
# MODELO
# ------------------------------------------------

rf_v2_purged = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    min_samples_leaf=10,
    min_samples_split=20,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("\nEntrenando...")

rf_v2_purged.fit(X_train, y_train)

print("Entrenamiento terminado.")


# ================================================================
# FUNCIÓN DE EVALUACIÓN
# ================================================================

def evaluar_modelo(nombre, X, y):

    pred = rf_v2_purged.predict(X)

    accuracy = accuracy_score(y, pred)
    balanced = balanced_accuracy_score(y, pred)
    macro_f1 = f1_score(
        y,
        pred,
        average="macro"
    )

    print("\n" + "=" * 75)
    print(nombre)
    print("=" * 75)

    print(f"Accuracy:          {accuracy:.4f}")
    print(f"Balanced Accuracy: {balanced:.4f}")
    print(f"Macro F1:          {macro_f1:.4f}")

    print("\nClassification Report:")

    print(
        classification_report(
            y,
            pred,
            labels=[0, 1, 2],
            target_names=[
                "NO OPERAR",
                "LONG",
                "SHORT"
            ],
            digits=4,
            zero_division=0
        )
    )

    cm = confusion_matrix(
        y,
        pred,
        labels=[0, 1, 2]
    )

    print("Matriz de confusión:")
    print(
        "            PRED NO  PRED LONG  PRED SHORT"
    )

    print(
        f"REAL NO    {cm[0,0]:8d}  {cm[0,1]:9d}  {cm[0,2]:10d}"
    )

    print(
        f"REAL LONG  {cm[1,0]:8d}  {cm[1,1]:9d}  {cm[1,2]:10d}"
    )

    print(
        f"REAL SHORT {cm[2,0]:8d}  {cm[2,1]:9d}  {cm[2,2]:10d}"
    )

    return pred


# ================================================================
# EVALUACIONES
# ================================================================

pred_train = evaluar_modelo(
    "TRAIN",
    X_train,
    y_train
)

pred_validation = evaluar_modelo(
    "VALIDATION",
    X_validation,
    y_validation
)

pred_oos = evaluar_modelo(
    "OOS",
    X_oos,
    y_oos
)

print("\n✓ Entrenamiento V2 purgado terminado.")

# ================================================================
# V2 — ANÁLISIS DE PROBABILIDADES
# ================================================================

import numpy as np
import pandas as pd

print("=" * 75)
print("V2 — ANÁLISIS DE PROBABILIDADES")
print("=" * 75)

# ------------------------------------------------
# PROBABILIDADES
# ------------------------------------------------

proba_train = rf_v2_purged.predict_proba(X_train)
proba_validation = rf_v2_purged.predict_proba(X_validation)
proba_oos = rf_v2_purged.predict_proba(X_oos)

print("\nOrden de clases del modelo:")
print(rf_v2_purged.classes_)

# ------------------------------------------------
# Crear DataFrames
# ------------------------------------------------

proba_val = pd.DataFrame(
    proba_validation,
    columns=["P_NO", "P_LONG", "P_SHORT"],
    index=validation.index
)

proba_oos_df = pd.DataFrame(
    proba_oos,
    columns=["P_NO", "P_LONG", "P_SHORT"],
    index=oos.index
)

# ------------------------------------------------
# Predicción y confianza
# ------------------------------------------------

proba_val["prediction"] = proba_val[
    ["P_NO", "P_LONG", "P_SHORT"]
].idxmax(axis=1)

proba_oos_df["prediction"] = proba_oos_df[
    ["P_NO", "P_LONG", "P_SHORT"]
].idxmax(axis=1)

proba_val["confidence"] = proba_val[
    ["P_NO", "P_LONG", "P_SHORT"]
].max(axis=1)

proba_oos_df["confidence"] = proba_oos_df[
    ["P_NO", "P_LONG", "P_SHORT"]
].max(axis=1)

# ------------------------------------------------
# Resumen general
# ------------------------------------------------

print("\nCONFIDENCIA VALIDATION:")
print(proba_val["confidence"].describe())

print("\nCONFIDENCIA OOS:")
print(proba_oos_df["confidence"].describe())

# ------------------------------------------------
# Distribución de confianza
# ------------------------------------------------

bins = [
    0.00,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    1.01
]

labels = [
    "<0.40",
    "0.40-0.50",
    "0.50-0.60",
    "0.60-0.70",
    "0.70-0.80",
    "0.80-0.90",
    ">=0.90"
]

proba_val["confidence_bin"] = pd.cut(
    proba_val["confidence"],
    bins=bins,
    labels=labels,
    right=False
)

proba_oos_df["confidence_bin"] = pd.cut(
    proba_oos_df["confidence"],
    bins=bins,
    labels=labels,
    right=False
)

print("\nDISTRIBUCIÓN CONFIDENCIA — VALIDATION:")
print(
    proba_val["confidence_bin"]
    .value_counts()
    .sort_index()
)

print("\nDISTRIBUCIÓN CONFIDENCIA — OOS:")
print(
    proba_oos_df["confidence_bin"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------
# Confianza por predicción — OOS
# ------------------------------------------------

print("\nCONFIDENCIA POR CLASE PREDICHA — OOS:")

print(
    proba_oos_df.groupby("prediction")[
        ["P_NO", "P_LONG", "P_SHORT", "confidence"]
    ].mean()
)

# ------------------------------------------------
# Guardar target real
# ------------------------------------------------

proba_val["actual"] = y_validation.values
proba_oos_df["actual"] = y_oos.values

print("\n✓ Análisis de probabilidades terminado.")

## 6. Diagnóstico en Validation

A partir de aquí, cualquier decisión nueva del proyecto debe desarrollarse usando TRAIN/VALIDATION. OOS permanece congelado.


In [ ]:
# ================================================================
# V2 — ANÁLISIS DE UMBRAL DE CONFIANZA
# SOLO VALIDATION
# ================================================================

from sklearn.metrics import precision_score, recall_score, f1_score

print("=" * 75)
print("V2 — ANÁLISIS DE CONFIANZA — VALIDATION")
print("=" * 75)

thresholds = [
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90
]

results = []

for threshold in thresholds:

    # ------------------------------------------------
    # Predicción original
    # ------------------------------------------------

    pred = proba_val[
        ["P_NO", "P_LONG", "P_SHORT"]
    ].idxmax(axis=1)

    confidence = proba_val["confidence"]

    # ------------------------------------------------
    # Solo señales LONG / SHORT con confianza >= umbral
    # ------------------------------------------------

    signal_mask = (
        (confidence >= threshold) &
        (pred != "P_NO")
    )

    signal_pred = pred[signal_mask]
    signal_actual = proba_val.loc[
        signal_mask,
        "actual"
    ]

    n_signals = len(signal_pred)

    if n_signals > 0:

        precision_long = precision_score(
            signal_actual == 1,
            signal_pred == "P_LONG",
            zero_division=0
        )

        precision_short = precision_score(
            signal_actual == 2,
            signal_pred == "P_SHORT",
            zero_division=0
        )

        # Precisión global:
        # ¿cuántas señales LONG/SHORT acertaron exactamente?
        precision_total = np.mean(
            (
                ((signal_pred == "P_LONG") & (signal_actual == 1)) |
                ((signal_pred == "P_SHORT") & (signal_actual == 2))
            )
        )

        long_signals = np.sum(
            signal_pred == "P_LONG"
        )

        short_signals = np.sum(
            signal_pred == "P_SHORT"
        )

        long_correct = np.sum(
            (signal_pred == "P_LONG") &
            (signal_actual == 1)
        )

        short_correct = np.sum(
            (signal_pred == "P_SHORT") &
            (signal_actual == 2)
        )

    else:

        precision_long = 0
        precision_short = 0
        precision_total = 0

        long_signals = 0
        short_signals = 0
        long_correct = 0
        short_correct = 0

    results.append({
        "threshold": threshold,
        "signals": n_signals,
        "LONG_signals": long_signals,
        "SHORT_signals": short_signals,
        "LONG_correct": long_correct,
        "SHORT_correct": short_correct,
        "precision_total": precision_total,
        "precision_LONG": precision_long,
        "precision_SHORT": precision_short
    })


results_df = pd.DataFrame(results)

# Porcentajes
results_df["signal_pct"] = (
    results_df["signals"] /
    len(proba_val) * 100
)

results_df["precision_total_pct"] = (
    results_df["precision_total"] * 100
)

results_df["precision_LONG_pct"] = (
    results_df["precision_LONG"] * 100
)

results_df["precision_SHORT_pct"] = (
    results_df["precision_SHORT"] * 100
)

print("\nRESULTADOS:")

print(
    results_df[
        [
            "threshold",
            "signals",
            "signal_pct",
            "LONG_signals",
            "SHORT_signals",
            "precision_total_pct",
            "precision_LONG_pct",
            "precision_SHORT_pct"
        ]
    ].to_string(index=False)
)

print("\n✓ Análisis realizado exclusivamente sobre VALIDATION.")

In [ ]:
# ================================================================
# V2 — ANÁLISIS DEL POTENCIAL REAL DE LAS SEÑALES
# SOLO VALIDATION
# ================================================================

print("=" * 75)
print("V2 — POTENCIAL REAL DE MOVIMIENTO — VALIDATION")
print("=" * 75)

HORIZON = 12

# Usamos los índices originales de validation
val_idx = validation.index

entry = df.loc[val_idx, "open"].values

future_high = (
    df["high"]
    .rolling(HORIZON)
    .max()
    .shift(-HORIZON + 1)
    .loc[val_idx]
    .values
)

future_low = (
    df["low"]
    .rolling(HORIZON)
    .min()
    .shift(-HORIZON + 1)
    .loc[val_idx]
    .values
)

# ------------------------------------------------
# R
# ------------------------------------------------

R_val = df.loc[val_idx, "v2_R"].values

# Movimiento favorable LONG
mfe_long = (
    (future_high - entry) / R_val
)

# Movimiento favorable SHORT
mfe_short = (
    (entry - future_low) / R_val
)

# ------------------------------------------------
# Resumen de niveles
# ------------------------------------------------

levels = [1.0, 1.5, 2.5, 3.8]

print("\nPROBABILIDAD DE ALCANZAR CADA NIVEL:")

for level in levels:

    long_hit = np.mean(
        mfe_long >= level
    ) * 100

    short_hit = np.mean(
        mfe_short >= level
    ) * 100

    print(
        f"{level:>4.1f}R → "
        f"LONG: {long_hit:6.2f}% | "
        f"SHORT: {short_hit:6.2f}%"
    )

# ------------------------------------------------
# Distribución del MFE
# ------------------------------------------------

print("\nMFE LONG (en R):")
print(
    pd.Series(mfe_long)
    .replace([np.inf, -np.inf], np.nan)
    .describe(
        percentiles=[
            .10,
            .25,
            .50,
            .75,
            .90,
            .95
        ]
    )
)

print("\nMFE SHORT (en R):")
print(
    pd.Series(mfe_short)
    .replace([np.inf, -np.inf], np.nan)
    .describe(
        percentiles=[
            .10,
            .25,
            .50,
            .75,
            .90,
            .95
        ]
    )
)

print("\n✓ Análisis de movimiento futuro terminado.")

## 7. Backtest V2 auditado — Validation

Se conserva una sola versión del backtest auditado. La copia duplicada del notebook original fue eliminada del flujo activo.


In [ ]:
# ============================================================
# V2 — BACKTEST VALIDATION — VERSIÓN AUDITADA
# ============================================================

import numpy as np
import pandas as pd

print("=" * 75)
print("V2 — BACKTEST AUDITADO — VALIDATION")
print("=" * 75)

# ============================================================
# COSTOS
# ============================================================

FEE_ENTRY = 0.0010
FEE_EXIT = 0.0010

SLIPPAGE_ENTRY = 0.0005
SLIPPAGE_EXIT = 0.0005

FEE_BUFFER = 0.0008

# ============================================================
# REGLAS
# ============================================================

R_LEVEL_1 = 1.0
R_LEVEL_2 = 1.5
R_LEVEL_3 = 2.5
R_LEVEL_4 = 3.8

MAX_HOLD = 12

# ============================================================
# PREDICCIONES VALIDATION
# ============================================================

pred_validation = rf_v2_purged.predict(X_validation)

validation_bt = df.loc[X_validation.index].copy()

validation_bt["prediction"] = pred_validation

signals = validation_bt[
    validation_bt["prediction"].isin([1, 2])
].copy()

print()
print(f"Señales totales: {len(signals):,}")
print(f"LONG: {(signals['prediction'] == 1).sum():,}")
print(f"SHORT: {(signals['prediction'] == 2).sum():,}")


# ============================================================
# SIMULADOR
# ============================================================

def simulate_trade_audited(df, signal_index, direction):

    signal_pos = df.index.get_loc(signal_index)

    # Necesitamos una vela posterior para entrar
    if signal_pos + 1 >= len(df):
        return None

    entry_pos = signal_pos + 1

    # Exactamente MAX_HOLD velas desde la entrada
    last_pos = min(
        entry_pos + MAX_HOLD - 1,
        len(df) - 1
    )

    signal_row = df.iloc[signal_pos]
    entry_row = df.iloc[entry_pos]

    # --------------------------------------------------------
    # R calculado en la vela de señal
    # --------------------------------------------------------

    R = float(signal_row["v2_R"])

    if not np.isfinite(R) or R <= 0:
        return None

    raw_entry = float(entry_row["open"])

    # --------------------------------------------------------
    # Entrada efectiva con fee + slippage
    # --------------------------------------------------------

    if direction == "LONG":

        entry_price = raw_entry * (
            1 + FEE_ENTRY + SLIPPAGE_ENTRY
        )

        initial_stop = entry_price - R

        tp_1 = entry_price + R_LEVEL_1 * R
        tp_2 = entry_price + R_LEVEL_2 * R
        tp_3 = entry_price + R_LEVEL_3 * R
        tp_4 = entry_price + R_LEVEL_4 * R

        breakeven_stop = entry_price * (
            1 - FEE_BUFFER
        )

    else:

        entry_price = raw_entry * (
            1 - FEE_ENTRY - SLIPPAGE_ENTRY
        )

        initial_stop = entry_price + R

        tp_1 = entry_price - R_LEVEL_1 * R
        tp_2 = entry_price - R_LEVEL_2 * R
        tp_3 = entry_price - R_LEVEL_3 * R
        tp_4 = entry_price - R_LEVEL_4 * R

        breakeven_stop = entry_price * (
            1 + FEE_BUFFER
        )

    # --------------------------------------------------------
    # Estado
    # --------------------------------------------------------

    current_stop = initial_stop

    reached_1R = False
    reached_1_5R = False
    reached_2_5R = False
    reached_3_8R = False

    exit_reason = None
    exit_price = None
    exit_pos = None

    max_R = 0.0

    # ========================================================
    # RECORRIDO
    # ========================================================

    for pos in range(entry_pos, last_pos + 1):

        row = df.iloc[pos]

        high = float(row["high"])
        low = float(row["low"])

        # ----------------------------------------------------
        # MFE de la operación
        # ----------------------------------------------------

        if direction == "LONG":
            favorable = high - entry_price
        else:
            favorable = entry_price - low

        max_R = max(max_R, favorable / R)

        # ----------------------------------------------------
        # IMPORTANTE:
        # usamos el stop que estaba activo AL INICIO de vela.
        # No aplicamos un stop recién movido dentro de esta misma
        # vela porque OHLC no permite conocer el orden intrabar.
        # ----------------------------------------------------

        stop_at_candle_start = current_stop

        # ----------------------------------------------------
        # STOP
        # ----------------------------------------------------

        if direction == "LONG":

            if low <= stop_at_candle_start:

                exit_price = stop_at_candle_start
                exit_pos = pos
                exit_reason = "STOP"
                break

        else:

            if high >= stop_at_candle_start:

                exit_price = stop_at_candle_start
                exit_pos = pos
                exit_reason = "STOP"
                break

        # ----------------------------------------------------
        # TARGETS
        # ----------------------------------------------------

        if direction == "LONG":

            # 3.8R = cierre total
            if high >= tp_4:

                exit_price = tp_4
                exit_pos = pos
                exit_reason = "TP_3.8R"
                reached_3_8R = True
                reached_2_5R = True
                reached_1_5R = True
                reached_1R = True
                break

            # 2.5R
            if high >= tp_3:
                reached_2_5R = True
                current_stop = tp_3

            # 1.5R
            if high >= tp_2:
                reached_1_5R = True
                current_stop = tp_2

            # 1R
            if high >= tp_1:
                reached_1R = True

                if not reached_1_5R:
                    current_stop = breakeven_stop

        else:

            # 3.8R
            if low <= tp_4:

                exit_price = tp_4
                exit_pos = pos
                exit_reason = "TP_3.8R"
                reached_3_8R = True
                reached_2_5R = True
                reached_1_5R = True
                reached_1R = True
                break

            # 2.5R
            if low <= tp_3:
                reached_2_5R = True
                current_stop = tp_3

            # 1.5R
            if low <= tp_2:
                reached_1_5R = True
                current_stop = tp_2

            # 1R
            if low <= tp_1:
                reached_1R = True

                if not reached_1_5R:
                    current_stop = breakeven_stop

    # ========================================================
    # TIME EXIT
    # ========================================================

    if exit_reason is None:

        exit_pos = last_pos

        exit_price = float(
            df.iloc[exit_pos]["close"]
        )

        exit_reason = "TIME_EXIT"

    # ========================================================
    # RETORNO NETO
    # ========================================================

    if direction == "LONG":

        effective_exit = exit_price * (
            1 - FEE_EXIT - SLIPPAGE_EXIT
        )

        net_return = (
            effective_exit / entry_price
        ) - 1

    else:

        effective_exit = exit_price * (
            1 + FEE_EXIT + SLIPPAGE_EXIT
        )

        net_return = (
            entry_price / effective_exit
        ) - 1

    result_R = net_return / (
        R / entry_price
    )

    return {
        "signal_index": signal_index,
        "direction": direction,
        "entry_price": entry_price,
        "R": R,
        "exit_price": exit_price,
        "exit_reason": exit_reason,
        "net_return": net_return,
        "result_R": result_R,
        "max_R": max_R,
        "reached_1R": reached_1R,
        "reached_1_5R": reached_1_5R,
        "reached_2_5R": reached_2_5R,
        "reached_3_8R": reached_3_8R
    }


# ============================================================
# EJECUTAR
# ============================================================

results = []

print()
print("Ejecutando simulación...")

for idx, row in signals.iterrows():

    direction = (
        "LONG"
        if row["prediction"] == 1
        else "SHORT"
    )

    result = simulate_trade_audited(
        validation_bt,
        idx,
        direction
    )

    if result is not None:
        results.append(result)


bt = pd.DataFrame(results)


# ============================================================
# RESULTADOS GENERALES
# ============================================================

print()
print("=" * 75)
print("RESULTADOS GENERALES")
print("=" * 75)

print(f"\nOperaciones:       {len(bt):,}")

win_rate = (
    (bt["result_R"] > 0).mean()
)

print(f"Win rate:          {win_rate:.2%}")

print(
    f"Retorno medio:     "
    f"{bt['net_return'].mean():.4%}"
)

print(
    f"Retorno mediano:   "
    f"{bt['net_return'].median():.4%}"
)

print(
    f"Resultado total R: "
    f"{bt['result_R'].sum():.2f}R"
)

print(
    f"Resultado medio R: "
    f"{bt['result_R'].mean():.4f}R"
)


# ============================================================
# PROFIT FACTOR
# ============================================================

gross_profit = bt.loc[
    bt["result_R"] > 0,
    "result_R"
].sum()

gross_loss = -bt.loc[
    bt["result_R"] < 0,
    "result_R"
].sum()

profit_factor = (
    gross_profit / gross_loss
    if gross_loss > 0
    else np.inf
)

print(
    f"Profit Factor:     "
    f"{profit_factor:.3f}"
)


# ============================================================
# RETORNO COMPUESTO
# ============================================================

compound_return = (
    (1 + bt["net_return"]).prod() - 1
)

print(
    f"Retorno compuesto: "
    f"{compound_return:.2%}"
)


# ============================================================
# DIRECCIÓN
# ============================================================

print()
print("=" * 75)
print("RESULTADOS POR DIRECCIÓN")
print("=" * 75)

for direction in ["LONG", "SHORT"]:

    subset = bt[
        bt["direction"] == direction
    ]

    print(f"\n{direction}")

    print(
        f"Operaciones:     "
        f"{len(subset):,}"
    )

    print(
        f"Win rate:        "
        f"{(subset['result_R'] > 0).mean():.2%}"
    )

    print(
        f"Retorno medio:   "
        f"{subset['net_return'].mean():.4%}"
    )

    print(
        f"Resultado R:     "
        f"{subset['result_R'].sum():.2f}R"
    )


# ============================================================
# MOTIVOS DE SALIDA
# ============================================================

print()
print("=" * 75)
print("MOTIVOS DE SALIDA")
print("=" * 75)

exit_summary = (
    bt.groupby("exit_reason")
      .agg(
          operaciones=("result_R", "count"),
          resultado_R=("result_R", "sum"),
          promedio_R=("result_R", "mean"),
          retorno_promedio=("net_return", "mean")
      )
      .sort_values(
          "operaciones",
          ascending=False
      )
)

print(exit_summary)


# ============================================================
# ALCANCE DE NIVELES
# ============================================================

print()
print("=" * 75)
print("ALCANCE DE NIVELES")
print("=" * 75)

total = len(bt)

for column, label in [
    ("reached_1R", "1R"),
    ("reached_1_5R", "1.5R"),
    ("reached_2_5R", "2.5R"),
    ("reached_3_8R", "3.8R")
]:

    count = bt[column].sum()

    print(
        f"{label:>5}: "
        f"{count:6,} "
        f"({count / total:.2%})"
    )


# ============================================================
# DISTRIBUCIÓN R
# ============================================================

print()
print("=" * 75)
print("DISTRIBUCIÓN RESULTADO R")
print("=" * 75)

print(
    bt["result_R"].describe(
        percentiles=[
            .10,
            .25,
            .50,
            .75,
            .90,
            .95
        ]
    )
)


# ============================================================
# DRAWDOWN
# ============================================================

equity = (
    1 + bt["net_return"]
).cumprod()

peak = equity.cummax()

drawdown = (
    equity / peak - 1
)

max_drawdown = drawdown.min()

print()
print("=" * 75)
print("DRAWDOWN")
print("=" * 75)

print(
    f"Max Drawdown: "
    f"{max_drawdown:.2%}"
)

print()
print("✓ BACKTEST V2 VALIDATION AUDITADO TERMINADO.")
print("✓ OOS NO FUE UTILIZADO.")

## 8. Análisis post-1R y auditoría completa


In [ ]:
# ============================================================
# V2 — VALIDATION — ANÁLISIS POST 1R
# ============================================================

print("=" * 75)
print("V2 — ANÁLISIS DE OPERACIONES QUE ALCANZAN 1R")
print("=" * 75)

# ------------------------------------------------------------
# 1. Operaciones que alcanzaron 1R
# ------------------------------------------------------------

post_1r = bt[bt["reached_1R"] == True].copy()

print()
print(f"Operaciones totales:       {len(bt):,}")
print(f"Alcanzaron 1R:             {len(post_1r):,}")
print(f"% que alcanza 1R:          {len(post_1r)/len(bt):.2%}")


# ------------------------------------------------------------
# 2. ¿Qué ocurrió después de alcanzar 1R?
# ------------------------------------------------------------

print()
print("=" * 75)
print("RESULTADO FINAL DE LAS OPERACIONES QUE ALCANZARON 1R")
print("=" * 75)

print(
    post_1r["exit_reason"]
    .value_counts()
)

print()
print(
    post_1r.groupby("exit_reason")
    .agg(
        operaciones=("result_R", "count"),
        resultado_R=("result_R", "sum"),
        promedio_R=("result_R", "mean"),
        retorno_promedio=("net_return", "mean"),
        max_R_promedio=("max_R", "mean")
    )
    .sort_values("operaciones", ascending=False)
)


# ------------------------------------------------------------
# 3. ¿Cuántas siguieron hasta cada nivel?
# ------------------------------------------------------------

print()
print("=" * 75)
print("PROGRESIÓN DESPUÉS DE 1R")
print("=" * 75)

n_1r = len(post_1r)

n_1_5r = post_1r["reached_1_5R"].sum()
n_2_5r = post_1r["reached_2_5R"].sum()
n_3_8r = post_1r["reached_3_8R"].sum()

print(
    f"1R:       {n_1r:6,} "
    f"(100.00%)"
)

print(
    f"1.5R:     {n_1_5r:6,} "
    f"({n_1_5r/n_1r:.2%} de las que alcanzaron 1R)"
)

print(
    f"2.5R:     {n_2_5r:6,} "
    f"({n_2_5r/n_1r:.2%} de las que alcanzaron 1R)"
)

print(
    f"3.8R:     {n_3_8r:6,} "
    f"({n_3_8r/n_1r:.2%} de las que alcanzaron 1R)"
)


# ------------------------------------------------------------
# 4. Resultado según máximo R alcanzado
# ------------------------------------------------------------

def max_r_group(x):

    if x >= 3.8:
        return ">= 3.8R"
    elif x >= 2.5:
        return "2.5R - <3.8R"
    elif x >= 1.5:
        return "1.5R - <2.5R"
    elif x >= 1.0:
        return "1R - <1.5R"
    else:
        return "<1R"


post_1r["max_R_group"] = post_1r["max_R"].apply(
    max_r_group
)

print()
print("=" * 75)
print("RESULTADO SEGÚN MÁXIMO MOVIMIENTO FAVORABLE")
print("=" * 75)

summary_max_r = (
    post_1r
    .groupby("max_R_group")
    .agg(
        operaciones=("result_R", "count"),
        resultado_R=("result_R", "sum"),
        promedio_R=("result_R", "mean"),
        win_rate=("result_R", lambda x: (x > 0).mean()),
        retorno_promedio=("net_return", "mean")
    )
)

orden = [
    "1R - <1.5R",
    "1.5R - <2.5R",
    "2.5R - <3.8R",
    ">= 3.8R"
]

print(
    summary_max_r.reindex(orden)
)


# ------------------------------------------------------------
# 5. ¿Cuántas operaciones que tocaron 1R terminaron negativas?
# ------------------------------------------------------------

negative_after_1r = post_1r[
    post_1r["result_R"] < 0
]

positive_after_1r = post_1r[
    post_1r["result_R"] > 0
]

zero_after_1r = post_1r[
    post_1r["result_R"] == 0
]

print()
print("=" * 75)
print("OPERACIONES QUE TOCARON 1R Y TERMINARON...")
print("=" * 75)

print(
    f"Negativas:  {len(negative_after_1r):,} "
    f"({len(negative_after_1r)/n_1r:.2%})"
)

print(
    f"Positivas:  {len(positive_after_1r):,} "
    f"({len(positive_after_1r)/n_1r:.2%})"
)

print(
    f"Neutras:    {len(zero_after_1r):,} "
    f"({len(zero_after_1r)/n_1r:.2%})"
)


# ------------------------------------------------------------
# 6. Separar LONG / SHORT
# ------------------------------------------------------------

print()
print("=" * 75)
print("POST 1R — LONG VS SHORT")
print("=" * 75)

for direction in ["LONG", "SHORT"]:

    subset = post_1r[
        post_1r["direction"] == direction
    ]

    print()
    print(direction)

    print(
        f"Operaciones: "
        f"{len(subset):,}"
    )

    print(
        f"Win rate: "
        f"{(subset['result_R'] > 0).mean():.2%}"
    )

    print(
        f"Resultado medio R: "
        f"{subset['result_R'].mean():.4f}R"
    )

    print(
        f"Retorno medio: "
        f"{subset['net_return'].mean():.4%}"
    )

    print(
        f"Alcanzaron 1.5R: "
        f"{subset['reached_1_5R'].mean():.2%}"
    )

    print(
        f"Alcanzaron 2.5R: "
        f"{subset['reached_2_5R'].mean():.2%}"
    )

    print(
        f"Alcanzaron 3.8R: "
        f"{subset['reached_3_8R'].mean():.2%}"
    )


print()
print("=" * 75)
print("✓ ANÁLISIS POST 1R TERMINADO")
print("=" * 75)

In [ ]:
# ============================================================
# V2 — VALIDATION — AUDITORÍA COMPLETA DE LA GESTIÓN
# ============================================================
# NO modifica:
# - modelo
# - features
# - target
# - TRAIN
# - OOS
#
# Usa únicamente el DataFrame `bt` generado por el
# BACKTEST AUDITADO.
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("V2 — VALIDATION — AUDITORÍA COMPLETA")
print("=" * 80)

# ============================================================
# 0. COMPROBACIÓN BÁSICA
# ============================================================

required_cols = [
    "direction",
    "R",
    "result_R",
    "net_return",
    "exit_reason",
    "max_R",
    "reached_1R",
    "reached_1_5R",
    "reached_2_5R",
    "reached_3_8R"
]

missing = [
    c for c in required_cols
    if c not in bt.columns
]

if missing:
    raise ValueError(
        f"Faltan columnas en bt: {missing}"
    )

work = bt.copy()

print(f"\nOperaciones analizadas: {len(work):,}")


# ============================================================
# 1. AUDITORÍA DE CONSISTENCIA DE NIVELES
# ============================================================

print()
print("=" * 80)
print("1 — AUDITORÍA DE CONSISTENCIA")
print("=" * 80)

# Una operación que tiene exit_reason TP_3.8R
# debería tener reached_3_8R = True.

tp38_reason = work["exit_reason"].eq("TP_3.8R")
tp38_flag = work["reached_3_8R"].astype(bool)

mismatch_tp38 = (tp38_reason != tp38_flag).sum()

print(
    f"\nTP_3.8R por motivo de salida: "
    f"{tp38_reason.sum():,}"
)

print(
    f"reached_3_8R = True: "
    f"{tp38_flag.sum():,}"
)

print(
    f"Discrepancias: "
    f"{mismatch_tp38:,}"
)

if mismatch_tp38 == 0:
    print("\n✓ TP_3.8R es consistente.")
else:
    print(
        "\n⚠️ Hay una discrepancia entre "
        "exit_reason y reached_3_8R."
    )


# ============================================================
# 2. GRUPOS: NUNCA 1R VS ALCANZÓ 1R
# ============================================================

work["grupo_1R"] = np.where(
    work["reached_1R"],
    "ALCANZO_1R",
    "NO_ALCANZO_1R"
)

print()
print("=" * 80)
print("2 — COMPARACIÓN 1R")
print("=" * 80)

comparison_1r = (
    work.groupby("grupo_1R")
    .agg(
        operaciones=("result_R", "count"),
        win_rate=(
            "result_R",
            lambda x: (x > 0).mean()
        ),
        retorno_medio=("net_return", "mean"),
        mediana_retorno=("net_return", "median"),
        resultado_medio_R=("result_R", "mean"),
        resultado_total_R=("result_R", "sum"),
        max_R_medio=("max_R", "mean")
    )
)

print()
print(comparison_1r)


# ============================================================
# 3. RESULTADO POR MOTIVO DE SALIDA
# ============================================================

print()
print("=" * 80)
print("3 — MOTIVOS DE SALIDA")
print("=" * 80)

exit_analysis = (
    work.groupby("exit_reason")
    .agg(
        operaciones=("result_R", "count"),
        win_rate=(
            "result_R",
            lambda x: (x > 0).mean()
        ),
        resultado_total_R=("result_R", "sum"),
        resultado_medio_R=("result_R", "mean"),
        retorno_medio=("net_return", "mean"),
        max_R_medio=("max_R", "mean")
    )
    .sort_values(
        "operaciones",
        ascending=False
    )
)

print()
print(exit_analysis)


# ============================================================
# 4. ANÁLISIS DE LAS QUE ALCANZAN 1R
# ============================================================

post_1r = work[
    work["reached_1R"]
].copy()

print()
print("=" * 80)
print("4 — OPERACIONES QUE ALCANZARON 1R")
print("=" * 80)

print(
    f"\nOperaciones: "
    f"{len(post_1r):,}"
)

print(
    f"Porcentaje del total: "
    f"{len(post_1r)/len(work):.2%}"
)

print(
    f"Win rate: "
    f"{(post_1r['result_R'] > 0).mean():.2%}"
)

print(
    f"Resultado medio: "
    f"{post_1r['result_R'].mean():.4f}R"
)

print(
    f"Retorno medio: "
    f"{post_1r['net_return'].mean():.4%}"
)


# ============================================================
# 5. PROGRESIÓN 1R → 1.5R → 2.5R → 3.8R
# ============================================================

print()
print("=" * 80)
print("5 — PROGRESIÓN DE NIVELES")
print("=" * 80)

levels = [
    ("1R", "reached_1R"),
    ("1.5R", "reached_1_5R"),
    ("2.5R", "reached_2_5R"),
    ("3.8R", "reached_3_8R")
]

for label, col in levels:

    n = work[col].sum()

    print(
        f"{label:>5}: "
        f"{n:6,} "
        f"({n/len(work):.2%} del total)"
    )

print("\nCondicional sobre las que alcanzaron 1R:")

for label, col in levels[1:]:

    n = post_1r[col].sum()

    print(
        f"{label:>5}: "
        f"{n:6,} "
        f"({n/len(post_1r):.2%} de las que alcanzaron 1R)"
    )


# ============================================================
# 6. RESULTADO SEGÚN MÁXIMO R
# ============================================================

def classify_max_r(x):

    if x >= 3.8:
        return ">= 3.8R"
    elif x >= 2.5:
        return "2.5R - <3.8R"
    elif x >= 1.5:
        return "1.5R - <2.5R"
    elif x >= 1.0:
        return "1R - <1.5R"
    else:
        return "<1R"


work["max_R_group"] = work["max_R"].apply(
    classify_max_r
)

order = [
    "<1R",
    "1R - <1.5R",
    "1.5R - <2.5R",
    "2.5R - <3.8R",
    ">= 3.8R"
]

max_r_analysis = (
    work.groupby("max_R_group")
    .agg(
        operaciones=("result_R", "count"),
        porcentaje=("result_R", "count"),
        win_rate=(
            "result_R",
            lambda x: (x > 0).mean()
        ),
        resultado_medio_R=("result_R", "mean"),
        resultado_total_R=("result_R", "sum"),
        retorno_medio=("net_return", "mean")
    )
    .reindex(order)
)

max_r_analysis["porcentaje"] /= len(work)

print()
print("=" * 80)
print("6 — RESULTADO SEGÚN MÁXIMO MOVIMIENTO FAVORABLE")
print("=" * 80)

print()
print(max_r_analysis)


# ============================================================
# 7. LONG VS SHORT
# ============================================================

print()
print("=" * 80)
print("7 — LONG VS SHORT")
print("=" * 80)

direction_analysis = (
    work.groupby("direction")
    .agg(
        operaciones=("result_R", "count"),
        win_rate=(
            "result_R",
            lambda x: (x > 0).mean()
        ),
        retorno_medio=("net_return", "mean"),
        resultado_medio_R=("result_R", "mean"),
        resultado_total_R=("result_R", "sum"),
        alcanza_1R=("reached_1R", "mean"),
        alcanza_1_5R=("reached_1_5R", "mean"),
        alcanza_2_5R=("reached_2_5R", "mean"),
        alcanza_3_8R=("reached_3_8R", "mean")
    )
)

print()
print(direction_analysis)


# ============================================================
# 8. LONG VS SHORT SOLO DESPUÉS DE 1R
# ============================================================

post_1r_direction = (
    post_1r.groupby("direction")
    .agg(
        operaciones=("result_R", "count"),
        win_rate=(
            "result_R",
            lambda x: (x > 0).mean()
        ),
        retorno_medio=("net_return", "mean"),
        resultado_medio_R=("result_R", "mean"),
        resultado_total_R=("result_R", "sum"),
        alcanza_1_5R=("reached_1_5R", "mean"),
        alcanza_2_5R=("reached_2_5R", "mean"),
        alcanza_3_8R=("reached_3_8R", "mean")
    )
)

print()
print("=" * 80)
print("8 — LONG VS SHORT DESPUÉS DE 1R")
print("=" * 80)

print()
print(post_1r_direction)


# ============================================================
# 9. EXPERIMENTO DIAGNÓSTICO:
#    "SI LLEGÓ A 1R, CIERRO EN 1R"
# ============================================================
#
# Esto NO modifica la estrategia.
# Es solamente un contrafactual para saber si el problema
# está en la gestión posterior a 1R.
#
# Para evitar inventar ejecución intrabar, usamos:
# resultado = aproximadamente +1R cuando alcanza 1R,
# manteniendo el costo implícito del backtest mediante el
# resultado_R observado de las operaciones que llegaron a 1R.
#
# Para un diagnóstico más exacto, hacemos también una versión
# teórica pura de +1R.
# ============================================================

n_post = len(post_1r)

if n_post > 0:

    theoretical_1R_result = (
        post_1r["R"] / post_1r["R"]
    )

    theoretical_total_R = (
        theoretical_1R_result.sum()
    )

    theoretical_avg_R = (
        theoretical_1R_result.mean()
    )

    print()
    print("=" * 80)
    print("9 — CONTRAFACTUAL: CERRAR TODA OPERACIÓN EN 1R")
    print("=" * 80)

    print(
        f"\nOperaciones que llegan a 1R: "
        f"{n_post:,}"
    )

    print(
        f"Resultado teórico por operación: "
        f"+1.000R"
    )

    print(
        f"Resultado total teórico: "
        f"{theoretical_total_R:.2f}R"
    )

    print(
        f"Resultado medio: "
        f"{theoretical_avg_R:.2f}R"
    )

    print(
        "\n⚠️ Esto es un diagnóstico teórico, "
        "NO un backtest ejecutable."
    )


# ============================================================
# 10. CONTRAFACTUAL CONSERVADOR:
#     ¿QUÉ PASA CON LAS QUE NO LLEGAN A 1R?
# ============================================================

no_1r = work[
    ~work["reached_1R"]
].copy()

print()
print("=" * 80)
print("10 — OPERACIONES QUE NUNCA LLEGARON A 1R")
print("=" * 80)

print(
    f"\nOperaciones: "
    f"{len(no_1r):,}"
)

print(
    f"Win rate: "
    f"{(no_1r['result_R'] > 0).mean():.2%}"
)

print(
    f"Resultado medio: "
    f"{no_1r['result_R'].mean():.4f}R"
)

print(
    f"Resultado total: "
    f"{no_1r['result_R'].sum():.2f}R"
)

print(
    f"Retorno medio: "
    f"{no_1r['net_return'].mean():.4%}"
)


# ============================================================
# 11. CUÁNTO DEL RESULTADO TOTAL VIENE DE CADA GRUPO
# ============================================================

total_R = work["result_R"].sum()

r_no_1 = no_1r["result_R"].sum()
r_yes_1 = post_1r["result_R"].sum()

print()
print("=" * 80)
print("11 — CONTRIBUCIÓN AL RESULTADO TOTAL")
print("=" * 80)

print(
    f"\nResultado total:       {total_R:.2f}R"
)

print(
    f"No alcanzan 1R:       {r_no_1:.2f}R"
)

print(
    f"Alcanzan 1R:          {r_yes_1:.2f}R"
)

print(
    f"Comprobación:         "
    f"{r_no_1 + r_yes_1:.2f}R"
)


# ============================================================
# 12. DIAGNÓSTICO FINAL AUTOMÁTICO
# ============================================================

print()
print("=" * 80)
print("12 — DIAGNÓSTICO")
print("=" * 80)

pct_1r = len(post_1r) / len(work)

if len(post_1r) > 0:

    post_1r_win = (
        post_1r["result_R"] > 0
    ).mean()

    post_1r_avg = (
        post_1r["result_R"].mean()
    )

else:

    post_1r_win = 0
    post_1r_avg = 0


print()

if pct_1r < 0.15:

    print(
        "→ POCAS OPERACIONES LLEGAN A 1R."
    )

    print(
        "  El principal problema parece estar "
        "en la calidad/frecuencia de las entradas."
    )

elif post_1r_win > 0.75:

    print(
        "→ LAS OPERACIONES QUE LLEGAN A 1R "
        "FUNCIONAN MUY BIEN."
    )

    print(
        "  Esto apunta principalmente a un "
        "problema de gestión y/o selección "
        "de las operaciones que nunca llegan a 1R."
    )

else:

    print(
        "→ EL COMPORTAMIENTO DESPUÉS DE 1R "
        "TAMBIÉN REQUIERE REVISIÓN."
    )


if r_yes_1 > 0 and r_no_1 < 0:

    print(
        "→ Las operaciones que alcanzan 1R "
        "aportan resultado positivo, mientras "
        "las que no llegan a 1R destruyen resultado."
    )

elif r_yes_1 < 0:

    print(
        "→ Incluso las operaciones que llegan "
        "a 1R tienen contribución negativa."
    )

else:

    print(
        "→ La contribución de los grupos no "
        "muestra una separación clara."
    )


print()
print("=" * 80)
print("✓ AUDITORÍA COMPLETA TERMINADA")
print("✓ OOS NO FUE UTILIZADO")
print("=" * 80)

## 9. Próximo paso del proyecto

**Pendiente:** analizar qué diferencia las señales de Validation que alcanzan 1R de las que no, usando únicamente información disponible antes de la entrada. Después se podrá diseñar un filtro con TRAIN/VALIDATION, congelarlo y recién entonces realizar la evaluación OOS final.

Variables candidatas ya disponibles: probabilidades por clase, margen de probabilidad, RSI, ATR%, volume ratio, MACD, distancia a EMA20/EMA50, retornos, volatilidad y dirección LONG/SHORT.


---
# Archivo histórico — NO ejecutar como parte del pipeline V2

Los bloques siguientes se conservan para trazabilidad. Contienen versiones anteriores, experimentos y evaluaciones OOS del V1. Pueden sobrescribir variables del pipeline activo si se ejecutan fuera de orden.


## Histórico A — Descarga de datos Binance/CCXT


In [ ]:
import ccxt
import pandas as pd
from datetime import datetime
import pandas as pd
import numpy as np
import vectorbt as vbt
import time

In [ ]:
# 1. Configuración de Binance Futuros con Rate Limit activo
exchange = ccxt.binance({
    'enableRateLimit': True,
    'options': {'defaultType': 'future'}
})

symbol = 'SOL/USDT:USDT'  # Formato estándar CCXT para contratos USDT-M
timeframe = '5m'
since = exchange.parse8601('2024-01-01T00:00:00Z')

all_candles = []
print(f"Iniciando descarga histórica de {symbol} ({timeframe})...")

while True:
    try:
        candles = exchange.fetch_ohlcv(symbol, timeframe=timeframe, since=since, limit=1000)
        if not candles:
            break
            
        all_candles.extend(candles)
        # Avanzar el timestamp al milisegundo siguiente de la última vela obtenida
        since = candles[-1][0] + 1
        
        # Pausa respetuosa de la API
        time.sleep(exchange.rateLimit / 1000)
        
        # Romper si llegamos al tiempo actual
        if candles[-1][0] >= exchange.milliseconds() - (5 * 60 * 1000):
            break
            
        if len(all_candles) % 20000 == 0:
            print(f"Descargadas {len(all_candles)} velas...")
            
    except Exception as e:
        print(f"Aviso de red / reintento: {e}")
        time.sleep(2)

# 2. Procesamiento a DataFrame
df = pd.DataFrame(all_candles, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
df.set_index('timestamp', inplace=True)

# Eliminar duplicados si los hubiera
df = df[~df.index.duplicated(keep='first')]

# Guardar a CSV
filename = 'SOLUSDT_5m_futures.csv'
df.to_csv(filename)
print(f"Descarga completada: {len(df)} velas guardadas en {filename}")

## Histórico B — Agente V3 basado en reglas


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# 1. CARGA DE DATOS HISTÓRICOS (5M)
# ==========================================================
print("Cargando base de datos histórica de 5 minutos...")
df = pd.read_csv('SOLUSDT_5m_futures.csv', index_col='timestamp', parse_dates=True)
df.sort_index(inplace=True)

close = df['close'].values
high = df['high'].values
low = df['low'].values
open_p = df['open'].values
vol = df['volume'].values
n_bars = len(df)

# ==========================================================
# 2. CONSTRUCCIÓN DE FILTROS MULTITEMPORALES (1H Y 15M)
# ==========================================================
print("Construyendo filtros macro 1H y 15M...")

# A. Macro 1H (Sesgo de dirección principal)
df_1h = df.resample('1h').agg({
    'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'volume': 'sum'
}).dropna()

h1_ema50 = df_1h['close'].ewm(span=50, adjust=False).mean()
h1_ema200 = df_1h['close'].ewm(span=200, adjust=False).mean()

# Desplazamiento de 1 barra en 1H para evitar look-ahead bias
h1_bull = ((df_1h['close'] > h1_ema200) & (h1_ema50 > h1_ema200)).shift(1)
h1_bear = ((df_1h['close'] < h1_ema200) & (h1_ema50 < h1_ema200)).shift(1)

macro_bullish = h1_bull.reindex(df.index, method='ffill').fillna(False).values
macro_bearish = h1_bear.reindex(df.index, method='ffill').fillna(False).values

# B. Tendencia intermedia 15M
df_15m = df.resample('15min').agg({
    'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'volume': 'sum'
}).dropna()
m15_ema50 = df_15m['close'].ewm(span=50, adjust=False).mean()
m15_bull = (df_15m['close'] >= m15_ema50).shift(1)
m15_bear = (df_15m['close'] <= m15_ema50).shift(1)

trend_15m_bull = m15_bull.reindex(df.index, method='ffill').fillna(False).values
trend_15m_bear = m15_bear.reindex(df.index, method='ffill').fillna(False).values

# ==========================================================
# 3. INDICADORES EN 5M Y SCORING DEL AGENTE V3
# ==========================================================
print("Calculando indicadores de 5M y matriz de scoring de 100 puntos...")

# Medias 5M
ema20 = df['close'].ewm(span=20, adjust=False).mean().values
ema50 = df['close'].ewm(span=50, adjust=False).mean().values

# VWAP diario aproximado
cum_vol = df['volume'].groupby(df.index.date).cumsum()
cum_vol_price = (df['close'] * df['volume']).groupby(df.index.date).cumsum()
vwap = (cum_vol_price / cum_vol).fillna(df['close']).values

# Bandas de Bollinger (20, 2)
bb_mid = df['close'].rolling(20).mean().values
bb_std = df['close'].rolling(20).std().values
bb_up = bb_mid + 2 * bb_std
bb_low = bb_mid - 2 * bb_std
bb_pct_b = np.where(bb_up != bb_low, (close - bb_low) / (bb_up - bb_low), 0.5)

# ATR (14) para Stop Loss dinámico
tr1 = high - low
tr2 = np.abs(high - np.roll(close, 1))
tr3 = np.abs(low - np.roll(close, 1))
tr = np.maximum(tr1, np.maximum(tr2, tr3))
tr[0] = tr1[0]
atr = pd.Series(tr, index=df.index).rolling(14).mean().values

# Volumen en retroceso (El bot real exige bajo volumen en pullback <= 1.2)
vol_sma20 = pd.Series(vol, index=df.index).rolling(20).mean().values
vol_ratio = np.where(vol_sma20 > 0, vol / vol_sma20, 1.0)
low_vol_pullback = vol_ratio <= 1.2

# StochRSI (14, 14, 3, 3)
delta = df['close'].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
rs = gain.rolling(14).mean() / loss.rolling(14).mean().replace(0, np.nan)
rsi_s = 100 - (100 / (1 + rs))
rsi_min = rsi_s.rolling(14).min()
rsi_max = rsi_s.rolling(14).max()

denom = rsi_max - rsi_min

stoch_raw = np.full_like(
    rsi_s,
    50.0,
    dtype=float
)

mask = denom != 0

stoch_raw[mask] = (
    (rsi_s[mask] - rsi_min[mask])
    / denom[mask]
) * 100

stoch_k = pd.Series(stoch_raw, index=df.index).rolling(3).mean().values
stoch_d = pd.Series(stoch_k, index=df.index).rolling(3).mean().values

prev_k = np.roll(stoch_k, 1)
prev_d = np.roll(stoch_d, 1)

# Cruces Estocásticos
stoch_cross_long = (prev_k < 30) & (stoch_k > stoch_d) & (prev_k <= prev_d)
stoch_cross_short = (prev_k > 70) & (stoch_k < stoch_d) & (prev_k >= prev_d)

# Velas de rechazo/confirmación
bullish_candle = (close >= open_p) | ((close - low) > (high - close))
bearish_candle = (close <= open_p) | ((high - close) > (close - low))

# Zonas de soporte y resistencia en pullback
pullback_support = (low <= ema20 * 1.002) | (bb_pct_b <= 0.38)
pullback_resistance = (high >= ema20 * 0.998) | (bb_pct_b >= 0.62)

# Filtro de Fin de Semana (Método A)
weekdays = df.index.weekday.values
hours = df.index.hour.values
is_weekend = ((weekdays == 4) & (hours >= 21)) | (weekdays == 5) | (weekdays == 6)

# SCORING LONG (100 pts)
score_long = np.zeros(n_bars)
score_long += np.where(macro_bullish, 20, 0)
score_long += np.where(trend_15m_bull, 20, 0)
score_long += np.where(close >= vwap, 10, 0)
score_long += np.where(ema20 >= ema50, 10, 0)
score_long += np.where(low_vol_pullback, 15, 0)
score_long += np.where(stoch_cross_long, 15, 0)
score_long += np.where(pullback_support & bullish_candle, 10, 0)

# SCORING SHORT (100 pts)
score_short = np.zeros(n_bars)
score_short += np.where(macro_bearish, 20, 0)
score_short += np.where(trend_15m_bear, 20, 0)
score_short += np.where(close <= vwap, 10, 0)
score_short += np.where(ema20 <= ema50, 10, 0)
score_short += np.where(low_vol_pullback, 15, 0)
score_short += np.where(stoch_cross_short, 15, 0)
score_short += np.where(pullback_resistance & bearish_candle, 10, 0)

# Señales de entrada
sig_long = (score_long >= 70) & (~is_weekend) & stoch_cross_long & pullback_support & bullish_candle
sig_short = (score_short >= 70) & (~is_weekend) & stoch_cross_short & pullback_resistance & bearish_candle

# Desplazar 1 barra para ejecutar al Open de la siguiente vela
entry_long = np.roll(sig_long, 1)
entry_long[0] = False
entry_short = np.roll(sig_short, 1)
entry_short[0] = False

# ==========================================================
# 4. SIMULADOR POSITION MANAGER REAL (BIDIRECCIONAL)
# ==========================================================
print("Ejecutando simulación de órdenes con Trailing Ladder y Break-Even...")
fee_rate = 0.0004
slippage_rate = 0.0002
round_cost = (fee_rate + slippage_rate) * 2

capital = 10000.0
initial_capital = capital
in_pos = False
side = None
entry_price = 0.0
initial_sl = 0.0
current_sl = 0.0
stage = 0
consecutive_losses = 0
cooldown_bars = 0

trades = []
equity_history = [capital]
dates_history = [df.index[0]]

for i in range(n_bars):
    if cooldown_bars > 0:
        cooldown_bars -= 1

    if in_pos:
        sl_updated = False
        
        # -----------------------------
        # GESTIÓN DE POSICIÓN LONG
        # -----------------------------
        if side == 'LONG':
            r = entry_price - initial_sl
            # 1. Break-Even (+1.0R): Sube Stop a entrada + fee buffer
            if high[i] >= entry_price + 1.0 * r and stage < 1:
                current_sl = entry_price * 1.0008
                stage = 1
                sl_updated = True
            # 2. TP1 (+1.5R): Sube Stop a TP1
            if high[i] >= entry_price + 1.5 * r and stage < 2:
                current_sl = entry_price + 1.5 * r
                stage = 2
                sl_updated = True
            # 3. TP2 (+2.5R): Sube Stop a TP2
            if high[i] >= entry_price + 2.5 * r and stage < 3:
                current_sl = entry_price + 2.5 * r
                stage = 3
                sl_updated = True
            # 4. TP3 (+3.8R): Salida total
            if high[i] >= entry_price + 3.8 * r:
                exit_p = entry_price + 3.8 * r
                raw_ret = (exit_p - entry_price) / entry_price
                net_ret = (raw_ret - round_cost) * 2.0
                trade_pnl = (capital * 0.35) * net_ret
                capital += trade_pnl
                trades.append({'side': 'LONG', 'ret': net_ret, 'pnl_usd': trade_pnl, 'status': 'WIN', 'stage': 4})
                consecutive_losses = 0
                in_pos = False
                equity_history.append(capital)
                dates_history.append(df.index[i])
                continue

            # Verificación de Stop Loss
            check_sl = initial_sl if sl_updated else current_sl
            if low[i] <= check_sl:
                exit_p = min(open_p[i], check_sl)
                raw_ret = (exit_p - entry_price) / entry_price
                net_ret = (raw_ret - round_cost) * 2.0
                trade_pnl = (capital * 0.35) * net_ret
                capital += trade_pnl
                status = 'WIN' if net_ret > 0.002 else ('BREAK_EVEN' if net_ret >= -0.002 else 'LOSS')
                if status == 'LOSS':
                    consecutive_losses += 1
                    if consecutive_losses >= 2:
                        cooldown_bars = 18  # 90 min de cooldown
                else:
                    consecutive_losses = 0
                trades.append({'side': 'LONG', 'ret': net_ret, 'pnl_usd': trade_pnl, 'status': status, 'stage': stage})
                in_pos = False
                equity_history.append(capital)
                dates_history.append(df.index[i])

        # -----------------------------
        # GESTIÓN DE POSICIÓN SHORT
        # -----------------------------
        elif side == 'SHORT':
            r = initial_sl - entry_price
            # 1. Break-Even (+1.0R hacia abajo): Baja Stop a entrada - fee buffer
            if low[i] <= entry_price - 1.0 * r and stage < 1:
                current_sl = entry_price * 0.9992
                stage = 1
                sl_updated = True
            # 2. TP1 (+1.5R hacia abajo): Baja Stop a TP1
            if low[i] <= entry_price - 1.5 * r and stage < 2:
                current_sl = entry_price - 1.5 * r
                stage = 2
                sl_updated = True
            # 3. TP2 (+2.5R hacia abajo): Baja Stop a TP2
            if low[i] <= entry_price - 2.5 * r and stage < 3:
                current_sl = entry_price - 2.5 * r
                stage = 3
                sl_updated = True
            # 4. TP3 (+3.8R hacia abajo): Salida total
            if low[i] <= entry_price - 3.8 * r:
                exit_p = entry_price - 3.8 * r
                raw_ret = (entry_price - exit_p) / entry_price
                net_ret = (raw_ret - round_cost) * 2.0
                trade_pnl = (capital * 0.35) * net_ret
                capital += trade_pnl
                trades.append({'side': 'SHORT', 'ret': net_ret, 'pnl_usd': trade_pnl, 'status': 'WIN', 'stage': 4})
                consecutive_losses = 0
                in_pos = False
                equity_history.append(capital)
                dates_history.append(df.index[i])
                continue

            # Verificación de Stop Loss para SHORT
            check_sl = initial_sl if sl_updated else current_sl
            if high[i] >= check_sl:
                exit_p = max(open_p[i], check_sl)
                raw_ret = (entry_price - exit_p) / entry_price
                net_ret = (raw_ret - round_cost) * 2.0
                trade_pnl = (capital * 0.35) * net_ret
                capital += trade_pnl
                status = 'WIN' if net_ret > 0.002 else ('BREAK_EVEN' if net_ret >= -0.002 else 'LOSS')
                if status == 'LOSS':
                    consecutive_losses += 1
                    if consecutive_losses >= 2:
                        cooldown_bars = 18  # 90 min de cooldown
                else:
                    consecutive_losses = 0
                trades.append({'side': 'SHORT', 'ret': net_ret, 'pnl_usd': trade_pnl, 'status': status, 'stage': stage})
                in_pos = False
                equity_history.append(capital)
                dates_history.append(df.index[i])

    else:
        # Apertura de posición si no hay cooldown
        if cooldown_bars == 0:
            if entry_long[i]:
                in_pos = True
                side = 'LONG'
                entry_price = open_p[i]
                sl_dist = max(entry_price * 0.015, min(atr[i] * 1.8, entry_price * 0.030))
                initial_sl = entry_price - sl_dist
                current_sl = initial_sl
                stage = 0
            elif entry_short[i]:
                in_pos = True
                side = 'SHORT'
                entry_price = open_p[i]
                sl_dist = max(entry_price * 0.015, min(atr[i] * 1.8, entry_price * 0.030))
                initial_sl = entry_price + sl_dist
                current_sl = initial_sl
                stage = 0

# ==========================================================
# 5. REPORTE FINAL DE MÉTRICAS CUANTITATIVAS
# ==========================================================
tdf = pd.DataFrame(trades)
print("\n" + "=" * 60)
print("📊 RESULTADOS DEL BACKTEST BIDIRECCIONAL - AGENTE V3 COMPLETO")
print("=" * 60)
print(f"Período evaluado: {df.index[0]} a {df.index[-1]}")
print(f"Capital Inicial: ${initial_capital:,.2f} | Capital Final: ${capital:,.2f}")
total_return = ((capital - initial_capital) / initial_capital) * 100
print(f"Rendimiento Total: {total_return:+.2f}%")
print(f"Total Operaciones: {len(tdf)}")

if len(tdf) > 0:
    longs = len(tdf[tdf['side'] == 'LONG'])
    shorts = len(tdf[tdf['side'] == 'SHORT'])
    wins = len(tdf[tdf['status'] == 'WIN'])
    be = len(tdf[tdf['status'] == 'BREAK_EVEN'])
    losses = len(tdf[tdf['status'] == 'LOSS'])
    
    wr_eff = (wins / (wins + losses)) * 100 if (wins + losses) > 0 else 0
    wr_global = (wins / len(tdf)) * 100
    
    print(f"  - Operaciones en LONG: {longs} | Operaciones en SHORT: {shorts}")
    print(f"  - Ganadas (WIN): {wins}")
    print(f"  - Empates (BREAK-EVEN): {be}")
    print(f"  - Perdidas (LOSS): {losses}")
    print(f"Win Rate Efectivo (excluyendo empates): {wr_eff:.1f}%")
    print(f"Win Rate Global: {wr_global:.1f}%")
    print(f"Retorno Medio por Operación: {tdf['ret'].mean()*100:+.2f}%")
    
    gross_profits = tdf[tdf['ret'] > 0]['ret'].sum()
    gross_losses = abs(tdf[tdf['ret'] < 0]['ret'].sum())
    profit_factor = gross_profits / gross_losses if gross_losses > 0 else np.nan
    print(f"Profit Factor: {profit_factor:.2f}")

    eq_series = pd.Series(equity_history, index=pd.to_datetime(dates_history))
    peak = eq_series.cummax()
    drawdown = (eq_series - peak) / peak
    print(f"Max Drawdown: {drawdown.min()*100:.2f}%")

    # Gráfico de curva de capital
    plt.figure(figsize=(12, 6))
    plt.plot(eq_series.index, eq_series.values, color='#00a65a', label='Curva de Capital (Long + Short)')
    plt.title('Curva de Capital - Agente v3 Completo Bidireccional (SOLUSDT 5M)')
    plt.xlabel('Fecha')
    plt.ylabel('Capital (USD)')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

## Histórico C — Random Forest / meta-modelo V1 completo


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import RandomForestClassifier


# ==========================================================
# CONFIGURACIÓN
# ==========================================================

DATA_FILE = 'SOLUSDT_5m_futures.csv'
MODEL_FILE = 'meta_model.joblib'

INITIAL_CAPITAL = 10_000.0

# 35% del capital como margen/exposición base
POSITION_FRACTION = 0.35

# Apalancamiento utilizado por la simulación
LEVERAGE = 2.0

# Comisión + slippage por operación completa
ROUND_COST = (0.0004 + 0.0002) * 2

# División temporal:
# 56% TRAIN
# 14% VALIDATION
# 30% TEST OOS
TRAIN_PCT = 0.56
VALIDATION_PCT = 0.70


# ==========================================================
# 1. CARGA DE DATOS
# ==========================================================

print("1. Cargando datos y construyendo indicadores...")

df = pd.read_csv(
    DATA_FILE,
    index_col='timestamp',
    parse_dates=True
)

df.sort_index(inplace=True)

required_cols = [
    'open',
    'high',
    'low',
    'close',
    'volume'
]

missing = [
    c for c in required_cols
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Faltan columnas obligatorias: {missing}"
    )

close = df['close'].values
high = df['high'].values
low = df['low'].values
open_p = df['open'].values
vol = df['volume'].values

n_bars = len(df)


# ==========================================================
# 2. OPEN INTEREST
# ==========================================================

oi_candidates = [
    'open_interest',
    'openInterest',
    'oi',
    'OI',
    'Open Interest',
    'open_interest_value'
]

oi_col = next(
    (
        c for c in oi_candidates
        if c in df.columns
    ),
    None
)

if oi_col is not None:

    oi = pd.to_numeric(
        df[oi_col],
        errors='coerce'
    ).ffill().fillna(0.0)

    # Cambio porcentual de OI.
    # Se utilizará desplazado una vela en las entradas.
    oi_change = (
        oi.pct_change()
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .values
    )

    print(
        f"Open Interest detectado: {oi_col}"
    )

else:

    print(
        "⚠️ No se encontró Open Interest "
        "en el CSV. Se utilizará 0.0."
    )

    oi_change = np.zeros(n_bars)


# ==========================================================
# 3. INDICADORES MULTITEMPORALES
# ==========================================================

# ----------------------------------------------------------
# MACRO 1H
# ----------------------------------------------------------

df_1h = (
    df.resample('1h')
    .agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    })
    .dropna()
)

h1_ema50 = (
    df_1h['close']
    .ewm(span=50, adjust=False)
    .mean()
)

h1_ema200 = (
    df_1h['close']
    .ewm(span=200, adjust=False)
    .mean()
)

macro_bullish = (
    (
        (df_1h['close'] > h1_ema200) &
        (h1_ema50 > h1_ema200)
    )
    .shift(1)
    .reindex(df.index, method='ffill')
    .fillna(False)
    .values
)

macro_bearish = (
    (
        (df_1h['close'] < h1_ema200) &
        (h1_ema50 < h1_ema200)
    )
    .shift(1)
    .reindex(df.index, method='ffill')
    .fillna(False)
    .values
)


# ----------------------------------------------------------
# TENDENCIA 15M
# ----------------------------------------------------------

df_15m = (
    df.resample('15min')
    .agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    })
    .dropna()
)

m15_ema50 = (
    df_15m['close']
    .ewm(span=50, adjust=False)
    .mean()
)

trend_15m_bull = (
    (df_15m['close'] >= m15_ema50)
    .shift(1)
    .reindex(df.index, method='ffill')
    .fillna(False)
    .values
)

trend_15m_bear = (
    (df_15m['close'] <= m15_ema50)
    .shift(1)
    .reindex(df.index, method='ffill')
    .fillna(False)
    .values
)


# ==========================================================
# 4. INDICADORES 5M
# ==========================================================

ema20 = (
    df['close']
    .ewm(span=20, adjust=False)
    .mean()
    .values
)

ema50 = (
    df['close']
    .ewm(span=50, adjust=False)
    .mean()
    .values
)


# ----------------------------------------------------------
# BOLLINGER
# ----------------------------------------------------------

bb_mid = (
    df['close']
    .rolling(20)
    .mean()
    .values
)

bb_std = (
    df['close']
    .rolling(20)
    .std()
    .values
)

bb_up = bb_mid + 2 * bb_std
bb_low = bb_mid - 2 * bb_std

bb_den = bb_up - bb_low

bb_pct_b = np.full(
    n_bars,
    0.5,
    dtype=float
)

bb_mask = (
    np.isfinite(bb_den) &
    (bb_den != 0)
)

bb_pct_b[bb_mask] = (
    (close[bb_mask] - bb_low[bb_mask])
    / bb_den[bb_mask]
)


# ----------------------------------------------------------
# ATR
# ----------------------------------------------------------

prev_close = np.roll(close, 1)

tr = np.maximum(
    high - low,
    np.maximum(
        np.abs(high - prev_close),
        np.abs(low - prev_close)
    )
)

tr[0] = high[0] - low[0]

atr = (
    pd.Series(
        tr,
        index=df.index
    )
    .rolling(14)
    .mean()
    .values
)


# ----------------------------------------------------------
# ADX
# ----------------------------------------------------------

prev_high = np.roll(high, 1)
prev_low = np.roll(low, 1)

up_m = high - prev_high
down_m = prev_low - low

p_dm = np.where(
    (up_m > down_m) & (up_m > 0),
    up_m,
    0.0
)

m_dm = np.where(
    (down_m > up_m) & (down_m > 0),
    down_m,
    0.0
)

safe_atr = np.where(
    np.isfinite(atr) & (atr > 0),
    atr,
    1.0
)

p_di = (
    100 *
    pd.Series(
        p_dm,
        index=df.index
    )
    .rolling(14)
    .mean()
    .values
    / safe_atr
)

m_di = (
    100 *
    pd.Series(
        m_dm,
        index=df.index
    )
    .rolling(14)
    .mean()
    .values
    / safe_atr
)

di_sum = np.where(
    np.isfinite(p_di + m_di) &
    ((p_di + m_di) > 0),
    p_di + m_di,
    1.0
)

adx = (
    pd.Series(
        100 * np.abs(p_di - m_di) / di_sum,
        index=df.index
    )
    .rolling(14)
    .mean()
    .fillna(20.0)
    .values
)


# ----------------------------------------------------------
# VOLUMEN
# ----------------------------------------------------------

vol_sma = (
    pd.Series(
        vol,
        index=df.index
    )
    .rolling(20)
    .mean()
    .values
)

vol_ratio = np.where(
    np.isfinite(vol_sma) & (vol_sma > 0),
    vol / vol_sma,
    1.0
)


# ==========================================================
# 5. RSI + STOCHRSI
# ==========================================================

delta = df['close'].diff()

gain = (
    delta
    .clip(lower=0)
    .rolling(14)
    .mean()
)

loss = (
    -delta
    .clip(upper=0)
    .rolling(14)
    .mean()
)

loss = loss.replace(0, np.nan)

rs = gain / loss

rsi_s = (
    100 - (100 / (1 + rs))
)

rsi_s = (
    rsi_s
    .replace([np.inf, -np.inf], np.nan)
    .fillna(50.0)
    .values
)

rsi_min = (
    pd.Series(
        rsi_s,
        index=df.index
    )
    .rolling(14)
    .min()
    .values
)

rsi_max = (
    pd.Series(
        rsi_s,
        index=df.index
    )
    .rolling(14)
    .max()
    .values
)

# ----------------------------------------------------------
# CORRECCIÓN DEL WARNING DEL STOCHRSI
# ----------------------------------------------------------

denom = rsi_max - rsi_min

stoch_raw = np.full(
    n_bars,
    50.0,
    dtype=float
)

stoch_mask = (
    np.isfinite(denom) &
    np.isfinite(rsi_s) &
    (denom != 0)
)

stoch_raw[stoch_mask] = (
    (
        rsi_s[stoch_mask]
        - rsi_min[stoch_mask]
    )
    / denom[stoch_mask]
) * 100

stoch_k = (
    pd.Series(
        stoch_raw,
        index=df.index
    )
    .rolling(3)
    .mean()
    .values
)

stoch_d = (
    pd.Series(
        stoch_k,
        index=df.index
    )
    .rolling(3)
    .mean()
    .values
)


# ==========================================================
# 6. SEÑALES
# ==========================================================

prev_k = np.roll(stoch_k, 1)
prev_d = np.roll(stoch_d, 1)

stoch_cross_long = (
    (prev_k < 30) &
    (stoch_k > stoch_d) &
    (prev_k <= prev_d)
)

stoch_cross_short = (
    (prev_k > 70) &
    (stoch_k < stoch_d) &
    (prev_k >= prev_d)
)


bullish_candle = (
    (close >= open_p) |
    ((close - low) > (high - close))
)

bearish_candle = (
    (close <= open_p) |
    ((high - close) > (close - low))
)


pullback_support = (
    (low <= ema20 * 1.002) |
    (bb_pct_b <= 0.38)
)

pullback_resistance = (
    (high >= ema20 * 0.998) |
    (bb_pct_b >= 0.62)
)


weekdays = df.index.weekday.values
hours = df.index.hour.values

is_weekend = (
    ((weekdays == 4) & (hours >= 21)) |
    (weekdays == 5) |
    (weekdays == 6)
)


# ----------------------------------------------------------
# FILTRO ADX
# ----------------------------------------------------------
#
# No imponemos un valor arbitrario.
# El ML será quien determine parte de la selección.
# ----------------------------------------------------------

sig_long = (
    macro_bullish &
    trend_15m_bull &
    stoch_cross_long &
    pullback_support &
    bullish_candle &
    (~is_weekend)
)

sig_short = (
    macro_bearish &
    trend_15m_bear &
    stoch_cross_short &
    pullback_resistance &
    bearish_candle &
    (~is_weekend)
)


# La señal se genera en la vela anterior.
# Entrada en OPEN de la vela siguiente.

entry_long = np.roll(sig_long, 1)
entry_long[0] = False

entry_short = np.roll(sig_short, 1)
entry_short[0] = False


# ==========================================================
# 7. FEATURES DEL META-MODELO
# ==========================================================

feature_cols = [
    'rsi_5m',
    'adx_5m',
    'atr_pct_5m',
    'volume_ratio_5m',
    'oi_change_5m',
    'hour_utc',
    'day_of_week',
    'side'
]


def get_features(i, is_long):
    """
    IMPORTANTE:
    La entrada ocurre en OPEN[i].

    Por eso TODAS las features utilizan i-1.
    No usamos información de cierre de la vela de entrada.
    """

    j = i - 1

    if j < 0:
        return None

    atr_value = atr[j]

    atr_pct = (
        atr_value / close[j]
        if np.isfinite(atr_value)
        and close[j] > 0
        else 0.0
    )

    feats = {
        'rsi_5m': float(rsi_s[j]),
        'adx_5m': float(adx[j]),
        'atr_pct_5m': float(atr_pct),
        'volume_ratio_5m': float(vol_ratio[j]),
        'oi_change_5m': float(oi_change[j]),
        'hour_utc': int(hours[j]),
        'day_of_week': int(weekdays[j]),
        'side': 1 if is_long else -1
    }

    # Protección NaN / Inf
    for k in feats:
        if not np.isfinite(feats[k]):
            feats[k] = 0.0

    return feats


# ==========================================================
# 8. SIMULADOR
# ==========================================================

def run_simulation(
    model=None,
    threshold=0.50,
    start_idx=0,
    end_idx=None
):

    if end_idx is None:
        end_idx = n_bars

    capital = INITIAL_CAPITAL

    in_pos = False
    side = None
    stage = 0

    consec_loss = 0
    cooldown = 0

    entry_p = 0.0
    init_sl = 0.0
    curr_sl = 0.0

    entry_date = None
    curr_feats = None

    records = []

    eq_curve = [capital]
    eq_dates = [df.index[start_idx]]

    for i in range(
        max(start_idx, 1),
        end_idx
    ):

        if cooldown > 0:
            cooldown -= 1


        # ==================================================
        # POSICIÓN ABIERTA
        # ==================================================

        if in_pos:

            r = abs(
                entry_p - init_sl
            )

            if r <= 0 or not np.isfinite(r):
                r = entry_p * 0.015


            # ----------------------------------------------
            # 1. PRIMERO COMPROBAMOS EL STOP ACTUAL
            #
            # Esto evita asumir que el máximo ocurrió antes
            # del mínimo dentro de la misma vela.
            # ----------------------------------------------

            hit_sl = (
                low[i] <= curr_sl
                if side == 'LONG'
                else high[i] >= curr_sl
            )

            if hit_sl:

                exit_p = curr_sl

                raw_ret = (
                    (exit_p - entry_p) / entry_p
                    if side == 'LONG'
                    else
                    (entry_p - exit_p) / entry_p
                )

                net_ret = (
                    raw_ret - ROUND_COST
                ) * LEVERAGE

                pnl = (
                    capital
                    * POSITION_FRACTION
                    * net_ret
                )

                capital += pnl

                status = (
                    'WIN'
                    if net_ret > 0.002
                    else
                    (
                        'BREAK_EVEN'
                        if net_ret >= -0.002
                        else 'LOSS'
                    )
                )

                y_label = (
                    1
                    if status in [
                        'WIN',
                        'BREAK_EVEN'
                    ]
                    else 0
                )

                if status == 'LOSS':

                    consec_loss += 1

                    if consec_loss >= 2:
                        cooldown = 18

                else:
                    consec_loss = 0

                records.append({
                    **curr_feats,
                    'entry_date': entry_date,
                    'ret': net_ret,
                    'pnl': pnl,
                    'status': status,
                    'y': y_label,
                    'date': df.index[i]
                })

                in_pos = False
                side = None
                entry_date = None
                curr_feats = None

                eq_curve.append(capital)
                eq_dates.append(df.index[i])

                continue


            # ----------------------------------------------
            # FAVORABLE MOVE
            # ----------------------------------------------

            favor = (
                high[i] - entry_p
                if side == 'LONG'
                else entry_p - low[i]
            )


            # ----------------------------------------------
            # TP3
            # ----------------------------------------------

            if favor >= 3.8 * r:

                net_ret = (
                    (3.8 * r / entry_p)
                    - ROUND_COST
                ) * LEVERAGE

                pnl = (
                    capital
                    * POSITION_FRACTION
                    * net_ret
                )

                capital += pnl

                records.append({
                    **curr_feats,
                    'entry_date': entry_date,
                    'ret': net_ret,
                    'pnl': pnl,
                    'status': 'WIN',
                    'y': 1,
                    'date': df.index[i]
                })

                in_pos = False
                side = None
                entry_date = None
                curr_feats = None
                consec_loss = 0

                eq_curve.append(capital)
                eq_dates.append(df.index[i])

                continue


            # ----------------------------------------------
            # TRAILING STOP
            #
            # El nuevo stop se aplica DESDE LA PRÓXIMA
            # vela. No lo usamos contra el mismo high/low.
            # ----------------------------------------------

            if favor >= 2.5 * r and stage < 3:

                curr_sl = (
                    entry_p + 2.5 * r
                    if side == 'LONG'
                    else entry_p - 2.5 * r
                )

                stage = 3

            elif favor >= 1.5 * r and stage < 2:

                curr_sl = (
                    entry_p + 1.5 * r
                    if side == 'LONG'
                    else entry_p - 1.5 * r
                )

                stage = 2

            elif favor >= 1.0 * r and stage < 1:

                curr_sl = (
                    entry_p * 1.0008
                    if side == 'LONG'
                    else entry_p * 0.9992
                )

                stage = 1


        # ==================================================
        # SIN POSICIÓN
        # ==================================================

        else:

            if cooldown == 0 and (
                entry_long[i] or
                entry_short[i]
            ):

                is_long = bool(
                    entry_long[i]
                )

                curr_feats = get_features(
                    i,
                    is_long
                )

                if curr_feats is None:
                    continue


                # ------------------------------------------
                # META-MODELO
                # ------------------------------------------

                allow_entry = True

                if model is not None:

                    feat_v = [
                        curr_feats[k]
                        for k in feature_cols
                    ]

                    class_1_idx = list(
                        model.classes_
                    ).index(1)

                    p_win = float(
                        model.predict_proba(
                            [feat_v]
                        )[0, class_1_idx]
                    )

                    if p_win < threshold:
                        allow_entry = False


                # ------------------------------------------
                # ENTRADA
                # ------------------------------------------

                if allow_entry:

                    in_pos = True

                    side = (
                        'LONG'
                        if is_long
                        else 'SHORT'
                    )

                    entry_p = open_p[i]

                    atr_value = atr[i - 1]

                    if (
                        not np.isfinite(
                            atr_value
                        )
                        or atr_value <= 0
                    ):
                        atr_value = (
                            entry_p * 0.015
                        )

                    sl_dist = max(
                        entry_p * 0.015,
                        min(
                            atr_value * 1.8,
                            entry_p * 0.030
                        )
                    )

                    init_sl = (
                        entry_p - sl_dist
                        if is_long
                        else entry_p + sl_dist
                    )

                    curr_sl = init_sl
                    stage = 0

                    entry_date = df.index[i]


    equity_curve = pd.Series(
        eq_curve,
        index=pd.to_datetime(
            eq_dates
        )
    )

    trades_df = pd.DataFrame(
        records
    )

    # ======================================================
    # MAX DRAWDOWN
    # ======================================================

    if len(equity_curve) > 0:

        running_max = (
            equity_curve
            .cummax()
        )

        drawdown = (
            equity_curve
            - running_max
        ) / running_max

        max_drawdown = (
            abs(drawdown.min())
            * 100
        )

    else:
        max_drawdown = 0.0


    return (
        trades_df,
        equity_curve,
        capital,
        max_drawdown
    )


# ==========================================================
# 9. MÉTRICAS
# ==========================================================

def get_metrics(
    tdf,
    capital,
    max_dd
):

    if tdf.empty:

        return {
            'operations': 0,
            'wins': 0,
            'be': 0,
            'losses': 0,
            'win_rate': 0.0,
            'pf': np.nan,
            'return': 0.0,
            'max_dd': max_dd,
            'avg_trade': 0.0,
            'max_loss_streak': 0
        }


    wins = (
        tdf['status'] == 'WIN'
    ).sum()

    be = (
        tdf['status'] == 'BREAK_EVEN'
    ).sum()

    losses = (
        tdf['status'] == 'LOSS'
    ).sum()


    denominator = wins + losses

    win_rate = (
        wins / denominator * 100
        if denominator > 0
        else 0
    )


    gross_profit = (
        tdf.loc[
            tdf['ret'] > 0,
            'ret'
        ].sum()
    )

    gross_loss = abs(
        tdf.loc[
            tdf['ret'] < 0,
            'ret'
        ].sum()
    )

    pf = (
        gross_profit / gross_loss
        if gross_loss > 0
        else np.nan
    )


    total_return = (
        (capital - INITIAL_CAPITAL)
        / INITIAL_CAPITAL
    ) * 100


    avg_trade = (
        tdf['ret'].mean()
        * 100
    )


    # Racha máxima de pérdidas
    is_loss = (
        tdf['status'] == 'LOSS'
    )

    max_streak = 0
    current_streak = 0

    for value in is_loss:

        if value:
            current_streak += 1
            max_streak = max(
                max_streak,
                current_streak
            )
        else:
            current_streak = 0


    return {
        'operations': len(tdf),
        'wins': int(wins),
        'be': int(be),
        'losses': int(losses),
        'win_rate': win_rate,
        'pf': pf,
        'return': total_return,
        'max_dd': max_dd,
        'avg_trade': avg_trade,
        'max_loss_streak': max_streak
    }


# ==========================================================
# 10. TRAIN / VALIDATION / TEST
# ==========================================================

print(
    "\n2. Preparando evaluación "
    "Train / Validation / OOS..."
)

train_end = int(
    n_bars * TRAIN_PCT
)

oos_start = int(
    n_bars * VALIDATION_PCT
)

print(
    f"TRAIN: {df.index[0]} -> "
    f"{df.index[train_end]}"
)

print(
    f"VALIDATION: {df.index[train_end]} -> "
    f"{df.index[oos_start]}"
)

print(
    f"TEST OOS: {df.index[oos_start]} -> "
    f"{df.index[-1]}"
)


# ==========================================================
# 11. GENERAR DATASET BASE
# ==========================================================

print(
    "\n3. Generando dataset "
    "sin Meta-Modelo..."
)

raw_tdf, raw_curve, raw_cap, raw_dd = run_simulation(
    model=None,
    start_idx=0,
    end_idx=oos_start
)

print(
    f"Trades pre-OOS: "
    f"{len(raw_tdf)}"
)


# ==========================================================
# 12. SEPARAR TRAIN / VALIDATION POR FECHA
# ==========================================================

train_tdf = raw_tdf[
    raw_tdf['date']
    < df.index[train_end]
].copy()

validation_tdf = raw_tdf[
    raw_tdf['date']
    >= df.index[train_end]
].copy()


print(
    f"Trades TRAIN: "
    f"{len(train_tdf)}"
)

print(
    f"Trades VALIDATION: "
    f"{len(validation_tdf)}"
)


# ==========================================================
# 13. ENTRENAR MODELO INICIAL
# ==========================================================

X_train = (
    train_tdf[feature_cols]
    .values
)

y_train = (
    train_tdf['y']
    .values
)


model_train = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=25,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model_train.fit(
    X_train,
    y_train
)


# ==========================================================
# 14. OPTIMIZAR THRESHOLD EN VALIDATION
# ==========================================================

print(
    "\n4. Buscando threshold "
    "óptimo SOLO en VALIDATION..."
)

thresholds = np.arange(
    0.50,
    0.711,
    0.02
)

validation_results = []


for threshold in thresholds:

    val_tdf, val_curve, val_cap, val_dd = run_simulation(
        model=model_train,
        threshold=float(threshold),
        start_idx=train_end,
        end_idx=oos_start
    )

    metrics = get_metrics(
        val_tdf,
        val_cap,
        val_dd
    )

    validation_results.append({
        'threshold': float(threshold),
        **metrics
    })


validation_df = pd.DataFrame(
    validation_results
)


# ----------------------------------------------------------
# CRITERIO DE SELECCIÓN
#
# Preferimos PF > 1.
# Entre esos modelos buscamos mejor retorno ajustado
# por drawdown.
# ----------------------------------------------------------

profitable = validation_df[
    validation_df['pf'] > 1.0
].copy()


if len(profitable) > 0:

    profitable['score'] = (
        profitable['return']
        / profitable['max_dd'].clip(
            lower=1.0
        )
    )

    best_row = profitable.loc[
        profitable['score'].idxmax()
    ]

else:

    # Si ningún threshold consigue PF > 1
    # en validation, seleccionamos el mejor
    # score disponible sin mirar OOS.

    validation_df['score'] = (
        validation_df['return']
        / validation_df['max_dd'].clip(
            lower=1.0
        )
    )

    best_row = validation_df.loc[
        validation_df['score'].idxmax()
    ]


best_threshold = float(
    best_row['threshold']
)


print(
    "\nResultados Validation:"
)

print(
    validation_df[
        [
            'threshold',
            'operations',
            'win_rate',
            'pf',
            'return',
            'max_dd'
        ]
    ].to_string(
        index=False,
        formatters={
            'threshold': '{:.2f}'.format,
            'win_rate': '{:.2f}%'.format,
            'pf': '{:.2f}'.format,
            'return': '{:+.2f}%'.format,
            'max_dd': '{:.2f}%'.format
        }
    )
)


print(
    f"\n🎯 Threshold seleccionado: "
    f"{best_threshold:.2f}"
)


# ==========================================================
# 15. MODELO FINAL
#
# Ahora que ya elegimos threshold usando validation,
# podemos entrenar el modelo con TODOS los trades
# anteriores al OOS.
# ==========================================================

pre_oos_tdf = raw_tdf[
    raw_tdf['date']
    < df.index[oos_start]
].copy()

X_final = (
    pre_oos_tdf[feature_cols]
    .values
)

y_final = (
    pre_oos_tdf['y']
    .values
)


meta_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=25,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

meta_model.fit(
    X_final,
    y_final
)


# ==========================================================
# 16. IMPORTANCIA DE VARIABLES
# ==========================================================

print(
    "\nImportancia de variables:"
)

for name, importance in sorted(
    zip(
        feature_cols,
        meta_model.feature_importances_
    ),
    key=lambda x: x[1],
    reverse=True
):

    print(
        f"  - {name:20s}: "
        f"{importance * 100:.2f}%"
    )


# ==========================================================
# 17. TEST OOS
# ==========================================================

print(
    "\n5. Ejecutando TEST OOS "
    "con modelo completamente congelado..."
)


# ----------------------------------------------------------
# BASE OOS
# ----------------------------------------------------------

test_raw_tdf, test_raw_curve, test_raw_cap, test_raw_dd = run_simulation(
    model=None,
    start_idx=oos_start,
    end_idx=n_bars
)


# ----------------------------------------------------------
# ML OOS
# ----------------------------------------------------------

test_ml_tdf, test_ml_curve, test_ml_cap, test_ml_dd = run_simulation(
    model=meta_model,
    threshold=best_threshold,
    start_idx=oos_start,
    end_idx=n_bars
)


# ==========================================================
# 18. MÉTRICAS FINALES
# ==========================================================

m_raw = get_metrics(
    test_raw_tdf,
    test_raw_cap,
    test_raw_dd
)

m_ml = get_metrics(
    test_ml_tdf,
    test_ml_cap,
    test_ml_dd
)


# ==========================================================
# 19. REPORTE
# ==========================================================

print("\n" + "=" * 105)

print(
    "📊 RESULTADO FINAL — TEST OOS"
)

print("=" * 105)

print(
    f"{'Métrica':<25} | "
    f"{'Sin ML':<18} | "
    f"{'Con ML':<18}"
)

print("-" * 105)

print(
    f"{'Operaciones':<25} | "
    f"{m_raw['operations']:<18} | "
    f"{m_ml['operations']:<18}"
)

print(
    f"{'Ganadas':<25} | "
    f"{m_raw['wins']:<18} | "
    f"{m_ml['wins']:<18}"
)

print(
    f"{'Empates':<25} | "
    f"{m_raw['be']:<18} | "
    f"{m_ml['be']:<18}"
)

print(
    f"{'Pérdidas':<25} | "
    f"{m_raw['losses']:<18} | "
    f"{m_ml['losses']:<18}"
)

print(
    f"{'Win Rate':<25} | "
    f"{m_raw['win_rate']:.2f}%{'':<13} | "
    f"{m_ml['win_rate']:.2f}%{'':<13}"
)

print(
    f"{'Profit Factor':<25} | "
    f"{m_raw['pf']:.2f}{'':<15} | "
    f"{m_ml['pf']:.2f}{'':<15}"
)

print(
    f"{'Rendimiento':<25} | "
    f"{m_raw['return']:+.2f}%{'':<12} | "
    f"{m_ml['return']:+.2f}%{'':<12}"
)

print(
    f"{'Maximum Drawdown':<25} | "
    f"{m_raw['max_dd']:.2f}%{'':<13} | "
    f"{m_ml['max_dd']:.2f}%{'':<13}"
)

print(
    f"{'Trade promedio':<25} | "
    f"{m_raw['avg_trade']:+.4f}%{'':<11} | "
    f"{m_ml['avg_trade']:+.4f}%{'':<11}"
)

print(
    f"{'Racha máxima LOSS':<25} | "
    f"{m_raw['max_loss_streak']:<18} | "
    f"{m_ml['max_loss_streak']:<18}"
)

print("=" * 105)


# ==========================================================
# 20. VALIDACIÓN DE RENTABILIDAD
# ==========================================================

print("\n" + "=" * 70)

if m_ml['return'] > 0:

    print(
        "🟢 TEST OOS POSITIVO"
    )

    print(
        f"El Meta-Modelo obtuvo "
        f"{m_ml['return']:+.2f}% "
        f"en datos no utilizados durante "
        f"el entrenamiento."
    )

else:

    print(
        "🔴 TEST OOS TODAVÍA NEGATIVO"
    )

    print(
        f"El resultado OOS fue "
        f"{m_ml['return']:+.2f}%."
    )

    print(
        "No se fuerza ningún parámetro para "
        "hacerlo positivo."
    )

print("=" * 70)


# ==========================================================
# 21. GUARDAR MODELO
# ==========================================================

joblib.dump(
    {
        'model': meta_model,
        'threshold': best_threshold,
        'feature_cols': feature_cols
    },
    MODEL_FILE
)

print(
    f"\n✅ Modelo guardado como: "
    f"{MODEL_FILE}"
)


# ==========================================================
# 22. GRÁFICO OOS
# ==========================================================

plt.figure(
    figsize=(13, 7)
)

plt.plot(
    test_raw_curve.index,
    test_raw_curve.values,
    alpha=0.65,
    label=(
        f'Sin ML '
        f'({m_raw["return"]:+.1f}%)'
    )
)

plt.plot(
    test_ml_curve.index,
    test_ml_curve.values,
    linewidth=2,
    label=(
        f'Con ML '
        f'({m_ml["return"]:+.1f}%)'
    )
)

plt.axhline(
    INITIAL_CAPITAL,
    linestyle='--',
    alpha=0.5,
    label='Capital inicial'
)

plt.title(
    'TEST OOS — SOLUSDT 5M — Meta-Labeling ML'
)

plt.xlabel('Fecha')
plt.ylabel('Capital (USD)')

plt.grid(
    True,
    alpha=0.3
)

plt.legend()

plt.tight_layout()

plt.show()

## Histórico D — Backtest V2 duplicado del notebook original


In [ ]:
# ============================================================
# V2 — BACKTEST VALIDATION — VERSIÓN AUDITADA
# ============================================================

import numpy as np
import pandas as pd

print("=" * 75)
print("V2 — BACKTEST AUDITADO — VALIDATION")
print("=" * 75)

# ============================================================
# COSTOS
# ============================================================

FEE_ENTRY = 0.0010
FEE_EXIT = 0.0010

SLIPPAGE_ENTRY = 0.0005
SLIPPAGE_EXIT = 0.0005

FEE_BUFFER = 0.0008

# ============================================================
# REGLAS
# ============================================================

R_LEVEL_1 = 1.0
R_LEVEL_2 = 1.5
R_LEVEL_3 = 2.5
R_LEVEL_4 = 3.8

MAX_HOLD = 12

# ============================================================
# PREDICCIONES VALIDATION
# ============================================================

pred_validation = rf_v2_purged.predict(X_validation)

validation_bt = df.loc[X_validation.index].copy()

validation_bt["prediction"] = pred_validation

signals = validation_bt[
    validation_bt["prediction"].isin([1, 2])
].copy()

print()
print(f"Señales totales: {len(signals):,}")
print(f"LONG: {(signals['prediction'] == 1).sum():,}")
print(f"SHORT: {(signals['prediction'] == 2).sum():,}")


# ============================================================
# SIMULADOR
# ============================================================

def simulate_trade_audited(df, signal_index, direction):

    signal_pos = df.index.get_loc(signal_index)

    # Necesitamos una vela posterior para entrar
    if signal_pos + 1 >= len(df):
        return None

    entry_pos = signal_pos + 1

    # Exactamente MAX_HOLD velas desde la entrada
    last_pos = min(
        entry_pos + MAX_HOLD - 1,
        len(df) - 1
    )

    signal_row = df.iloc[signal_pos]
    entry_row = df.iloc[entry_pos]

    # --------------------------------------------------------
    # R calculado en la vela de señal
    # --------------------------------------------------------

    R = float(signal_row["v2_R"])

    if not np.isfinite(R) or R <= 0:
        return None

    raw_entry = float(entry_row["open"])

    # --------------------------------------------------------
    # Entrada efectiva con fee + slippage
    # --------------------------------------------------------

    if direction == "LONG":

        entry_price = raw_entry * (
            1 + FEE_ENTRY + SLIPPAGE_ENTRY
        )

        initial_stop = entry_price - R

        tp_1 = entry_price + R_LEVEL_1 * R
        tp_2 = entry_price + R_LEVEL_2 * R
        tp_3 = entry_price + R_LEVEL_3 * R
        tp_4 = entry_price + R_LEVEL_4 * R

        breakeven_stop = entry_price * (
            1 - FEE_BUFFER
        )

    else:

        entry_price = raw_entry * (
            1 - FEE_ENTRY - SLIPPAGE_ENTRY
        )

        initial_stop = entry_price + R

        tp_1 = entry_price - R_LEVEL_1 * R
        tp_2 = entry_price - R_LEVEL_2 * R
        tp_3 = entry_price - R_LEVEL_3 * R
        tp_4 = entry_price - R_LEVEL_4 * R

        breakeven_stop = entry_price * (
            1 + FEE_BUFFER
        )

    # --------------------------------------------------------
    # Estado
    # --------------------------------------------------------

    current_stop = initial_stop

    reached_1R = False
    reached_1_5R = False
    reached_2_5R = False
    reached_3_8R = False

    exit_reason = None
    exit_price = None
    exit_pos = None

    max_R = 0.0

    # ========================================================
    # RECORRIDO
    # ========================================================

    for pos in range(entry_pos, last_pos + 1):

        row = df.iloc[pos]

        high = float(row["high"])
        low = float(row["low"])

        # ----------------------------------------------------
        # MFE de la operación
        # ----------------------------------------------------

        if direction == "LONG":
            favorable = high - entry_price
        else:
            favorable = entry_price - low

        max_R = max(max_R, favorable / R)

        # ----------------------------------------------------
        # IMPORTANTE:
        # usamos el stop que estaba activo AL INICIO de vela.
        # No aplicamos un stop recién movido dentro de esta misma
        # vela porque OHLC no permite conocer el orden intrabar.
        # ----------------------------------------------------

        stop_at_candle_start = current_stop

        # ----------------------------------------------------
        # STOP
        # ----------------------------------------------------

        if direction == "LONG":

            if low <= stop_at_candle_start:

                exit_price = stop_at_candle_start
                exit_pos = pos
                exit_reason = "STOP"
                break

        else:

            if high >= stop_at_candle_start:

                exit_price = stop_at_candle_start
                exit_pos = pos
                exit_reason = "STOP"
                break

        # ----------------------------------------------------
        # TARGETS
        # ----------------------------------------------------

        if direction == "LONG":

            # 3.8R = cierre total
            if high >= tp_4:

                exit_price = tp_4
                exit_pos = pos
                exit_reason = "TP_3.8R"
                reached_3_8R = True
                reached_2_5R = True
                reached_1_5R = True
                reached_1R = True
                break

            # 2.5R
            if high >= tp_3:
                reached_2_5R = True
                current_stop = tp_3

            # 1.5R
            if high >= tp_2:
                reached_1_5R = True
                current_stop = tp_2

            # 1R
            if high >= tp_1:
                reached_1R = True

                if not reached_1_5R:
                    current_stop = breakeven_stop

        else:

            # 3.8R
            if low <= tp_4:

                exit_price = tp_4
                exit_pos = pos
                exit_reason = "TP_3.8R"
                reached_3_8R = True
                reached_2_5R = True
                reached_1_5R = True
                reached_1R = True
                break

            # 2.5R
            if low <= tp_3:
                reached_2_5R = True
                current_stop = tp_3

            # 1.5R
            if low <= tp_2:
                reached_1_5R = True
                current_stop = tp_2

            # 1R
            if low <= tp_1:
                reached_1R = True

                if not reached_1_5R:
                    current_stop = breakeven_stop

    # ========================================================
    # TIME EXIT
    # ========================================================

    if exit_reason is None:

        exit_pos = last_pos

        exit_price = float(
            df.iloc[exit_pos]["close"]
        )

        exit_reason = "TIME_EXIT"

    # ========================================================
    # RETORNO NETO
    # ========================================================

    if direction == "LONG":

        effective_exit = exit_price * (
            1 - FEE_EXIT - SLIPPAGE_EXIT
        )

        net_return = (
            effective_exit / entry_price
        ) - 1

    else:

        effective_exit = exit_price * (
            1 + FEE_EXIT + SLIPPAGE_EXIT
        )

        net_return = (
            entry_price / effective_exit
        ) - 1

    result_R = net_return / (
        R / entry_price
    )

    return {
        "signal_index": signal_index,
        "direction": direction,
        "entry_price": entry_price,
        "R": R,
        "exit_price": exit_price,
        "exit_reason": exit_reason,
        "net_return": net_return,
        "result_R": result_R,
        "max_R": max_R,
        "reached_1R": reached_1R,
        "reached_1_5R": reached_1_5R,
        "reached_2_5R": reached_2_5R,
        "reached_3_8R": reached_3_8R
    }


# ============================================================
# EJECUTAR
# ============================================================

results = []

print()
print("Ejecutando simulación...")

for idx, row in signals.iterrows():

    direction = (
        "LONG"
        if row["prediction"] == 1
        else "SHORT"
    )

    result = simulate_trade_audited(
        validation_bt,
        idx,
        direction
    )

    if result is not None:
        results.append(result)


bt = pd.DataFrame(results)


# ============================================================
# RESULTADOS GENERALES
# ============================================================

print()
print("=" * 75)
print("RESULTADOS GENERALES")
print("=" * 75)

print(f"\nOperaciones:       {len(bt):,}")

win_rate = (
    (bt["result_R"] > 0).mean()
)

print(f"Win rate:          {win_rate:.2%}")

print(
    f"Retorno medio:     "
    f"{bt['net_return'].mean():.4%}"
)

print(
    f"Retorno mediano:   "
    f"{bt['net_return'].median():.4%}"
)

print(
    f"Resultado total R: "
    f"{bt['result_R'].sum():.2f}R"
)

print(
    f"Resultado medio R: "
    f"{bt['result_R'].mean():.4f}R"
)


# ============================================================
# PROFIT FACTOR
# ============================================================

gross_profit = bt.loc[
    bt["result_R"] > 0,
    "result_R"
].sum()

gross_loss = -bt.loc[
    bt["result_R"] < 0,
    "result_R"
].sum()

profit_factor = (
    gross_profit / gross_loss
    if gross_loss > 0
    else np.inf
)

print(
    f"Profit Factor:     "
    f"{profit_factor:.3f}"
)


# ============================================================
# RETORNO COMPUESTO
# ============================================================

compound_return = (
    (1 + bt["net_return"]).prod() - 1
)

print(
    f"Retorno compuesto: "
    f"{compound_return:.2%}"
)


# ============================================================
# DIRECCIÓN
# ============================================================

print()
print("=" * 75)
print("RESULTADOS POR DIRECCIÓN")
print("=" * 75)

for direction in ["LONG", "SHORT"]:

    subset = bt[
        bt["direction"] == direction
    ]

    print(f"\n{direction}")

    print(
        f"Operaciones:     "
        f"{len(subset):,}"
    )

    print(
        f"Win rate:        "
        f"{(subset['result_R'] > 0).mean():.2%}"
    )

    print(
        f"Retorno medio:   "
        f"{subset['net_return'].mean():.4%}"
    )

    print(
        f"Resultado R:     "
        f"{subset['result_R'].sum():.2f}R"
    )


# ============================================================
# MOTIVOS DE SALIDA
# ============================================================

print()
print("=" * 75)
print("MOTIVOS DE SALIDA")
print("=" * 75)

exit_summary = (
    bt.groupby("exit_reason")
      .agg(
          operaciones=("result_R", "count"),
          resultado_R=("result_R", "sum"),
          promedio_R=("result_R", "mean"),
          retorno_promedio=("net_return", "mean")
      )
      .sort_values(
          "operaciones",
          ascending=False
      )
)

print(exit_summary)


# ============================================================
# ALCANCE DE NIVELES
# ============================================================

print()
print("=" * 75)
print("ALCANCE DE NIVELES")
print("=" * 75)

total = len(bt)

for column, label in [
    ("reached_1R", "1R"),
    ("reached_1_5R", "1.5R"),
    ("reached_2_5R", "2.5R"),
    ("reached_3_8R", "3.8R")
]:

    count = bt[column].sum()

    print(
        f"{label:>5}: "
        f"{count:6,} "
        f"({count / total:.2%})"
    )


# ============================================================
# DISTRIBUCIÓN R
# ============================================================

print()
print("=" * 75)
print("DISTRIBUCIÓN RESULTADO R")
print("=" * 75)

print(
    bt["result_R"].describe(
        percentiles=[
            .10,
            .25,
            .50,
            .75,
            .90,
            .95
        ]
    )
)


# ============================================================
# DRAWDOWN
# ============================================================

equity = (
    1 + bt["net_return"]
).cumprod()

peak = equity.cummax()

drawdown = (
    equity / peak - 1
)

max_drawdown = drawdown.min()

print()
print("=" * 75)
print("DRAWDOWN")
print("=" * 75)

print(
    f"Max Drawdown: "
    f"{max_drawdown:.2%}"
)

print()
print("✓ BACKTEST V2 VALIDATION AUDITADO TERMINADO.")
print("✓ OOS NO FUE UTILIZADO.")

# ============================================================
# V2 — VALIDATION — ANÁLISIS POST 1R
# ============================================================

print("=" * 75)
print("V2 — ANÁLISIS DE OPERACIONES QUE ALCANZAN 1R")
print("=" * 75)

# ------------------------------------------------------------
# 1. Operaciones que alcanzaron 1R
# ------------------------------------------------------------

post_1r = bt[bt["reached_1R"] == True].copy()

print()
print(f"Operaciones totales:       {len(bt):,}")
print(f"Alcanzaron 1R:             {len(post_1r):,}")
print(f"% que alcanza 1R:          {len(post_1r)/len(bt):.2%}")


# ------------------------------------------------------------
# 2. ¿Qué ocurrió después de alcanzar 1R?
# ------------------------------------------------------------

print()
print("=" * 75)
print("RESULTADO FINAL DE LAS OPERACIONES QUE ALCANZARON 1R")
print("=" * 75)

print(
    post_1r["exit_reason"]
    .value_counts()
)

print()
print(
    post_1r.groupby("exit_reason")
    .agg(
        operaciones=("result_R", "count"),
        resultado_R=("result_R", "sum"),
        promedio_R=("result_R", "mean"),
        retorno_promedio=("net_return", "mean"),
        max_R_promedio=("max_R", "mean")
    )
    .sort_values("operaciones", ascending=False)
)


# ------------------------------------------------------------
# 3. ¿Cuántas siguieron hasta cada nivel?
# ------------------------------------------------------------

print()
print("=" * 75)
print("PROGRESIÓN DESPUÉS DE 1R")
print("=" * 75)

n_1r = len(post_1r)

n_1_5r = post_1r["reached_1_5R"].sum()
n_2_5r = post_1r["reached_2_5R"].sum()
n_3_8r = post_1r["reached_3_8R"].sum()

print(
    f"1R:       {n_1r:6,} "
    f"(100.00%)"
)

print(
    f"1.5R:     {n_1_5r:6,} "
    f"({n_1_5r/n_1r:.2%} de las que alcanzaron 1R)"
)

print(
    f"2.5R:     {n_2_5r:6,} "
    f"({n_2_5r/n_1r:.2%} de las que alcanzaron 1R)"
)

print(
    f"3.8R:     {n_3_8r:6,} "
    f"({n_3_8r/n_1r:.2%} de las que alcanzaron 1R)"
)


# ------------------------------------------------------------
# 4. Resultado según máximo R alcanzado
# ------------------------------------------------------------

def max_r_group(x):

    if x >= 3.8:
        return ">= 3.8R"
    elif x >= 2.5:
        return "2.5R - <3.8R"
    elif x >= 1.5:
        return "1.5R - <2.5R"
    elif x >= 1.0:
        return "1R - <1.5R"
    else:
        return "<1R"


post_1r["max_R_group"] = post_1r["max_R"].apply(
    max_r_group
)

print()
print("=" * 75)
print("RESULTADO SEGÚN MÁXIMO MOVIMIENTO FAVORABLE")
print("=" * 75)

summary_max_r = (
    post_1r
    .groupby("max_R_group")
    .agg(
        operaciones=("result_R", "count"),
        resultado_R=("result_R", "sum"),
        promedio_R=("result_R", "mean"),
        win_rate=("result_R", lambda x: (x > 0).mean()),
        retorno_promedio=("net_return", "mean")
    )
)

orden = [
    "1R - <1.5R",
    "1.5R - <2.5R",
    "2.5R - <3.8R",
    ">= 3.8R"
]

print(
    summary_max_r.reindex(orden)
)


# ------------------------------------------------------------
# 5. ¿Cuántas operaciones que tocaron 1R terminaron negativas?
# ------------------------------------------------------------

negative_after_1r = post_1r[
    post_1r["result_R"] < 0
]

positive_after_1r = post_1r[
    post_1r["result_R"] > 0
]

zero_after_1r = post_1r[
    post_1r["result_R"] == 0
]

print()
print("=" * 75)
print("OPERACIONES QUE TOCARON 1R Y TERMINARON...")
print("=" * 75)

print(
    f"Negativas:  {len(negative_after_1r):,} "
    f"({len(negative_after_1r)/n_1r:.2%})"
)

print(
    f"Positivas:  {len(positive_after_1r):,} "
    f"({len(positive_after_1r)/n_1r:.2%})"
)

print(
    f"Neutras:    {len(zero_after_1r):,} "
    f"({len(zero_after_1r)/n_1r:.2%})"
)


# ------------------------------------------------------------
# 6. Separar LONG / SHORT
# ------------------------------------------------------------

print()
print("=" * 75)
print("POST 1R — LONG VS SHORT")
print("=" * 75)

for direction in ["LONG", "SHORT"]:

    subset = post_1r[
        post_1r["direction"] == direction
    ]

    print()
    print(direction)

    print(
        f"Operaciones: "
        f"{len(subset):,}"
    )

    print(
        f"Win rate: "
        f"{(subset['result_R'] > 0).mean():.2%}"
    )

    print(
        f"Resultado medio R: "
        f"{subset['result_R'].mean():.4f}R"
    )

    print(
        f"Retorno medio: "
        f"{subset['net_return'].mean():.4%}"
    )

    print(
        f"Alcanzaron 1.5R: "
        f"{subset['reached_1_5R'].mean():.2%}"
    )

    print(
        f"Alcanzaron 2.5R: "
        f"{subset['reached_2_5R'].mean():.2%}"
    )

    print(
        f"Alcanzaron 3.8R: "
        f"{subset['reached_3_8R'].mean():.2%}"
    )


print()
print("=" * 75)
print("✓ ANÁLISIS POST 1R TERMINADO")
print("=" * 75)

## Histórico E — Diagnósticos, calibración y OOS del modelo binario V1


In [ ]:
# ============================================================
# ANÁLISIS DE PRECISIÓN POR UMBRAL
# ============================================================

print("=" * 70)
print("PRECISIÓN SEGÚN UMBRAL DE PROBABILIDAD")
print("=" * 70)

thresholds = [
    0.50,
    0.52,
    0.55,
    0.58,
    0.60,
    0.62,
    0.65,
    0.70
]

for threshold in thresholds:

    mask = oos_proba >= threshold

    samples = mask.sum()

    if samples > 0:

        accuracy = (
            y_oos.values[mask] == 1
        ).mean()

        coverage = (
            samples / len(oos)
        ) * 100

        print(
            f"\nUmbral >= {threshold:.2f}"
        )

        print(
            f"Casos: {samples:,}"
        )

        print(
            f"Accuracy: {accuracy:.2%}"
        )

        print(
            f"Coverage: {coverage:.2f}%"
        )

    else:

        print(
            f"\nUmbral >= {threshold:.2f}: "
            f"sin casos"
        )

In [ ]:
# ============================================================
# CALIBRACIÓN REAL DEL MODELO
# ============================================================

print("=" * 70)
print("CALIBRACIÓN DE PROBABILIDADES OOS")
print("=" * 70)

calibration = pd.DataFrame({
    "probability": oos_proba,
    "target": y_oos.values
})

# Dividimos en intervalos
calibration["bucket"] = pd.cut(
    calibration["probability"],
    bins=[
        0.0,
        0.40,
        0.45,
        0.50,
        0.55,
        0.60,
        0.65,
        0.70,
        0.80,
        1.00
    ],
    include_lowest=True
)

summary = (
    calibration
    .groupby("bucket", observed=True)
    .agg(
        casos=("target", "size"),
        positivos_reales=("target", "sum"),
        tasa_real=("target", "mean"),
        probabilidad_media=("probability", "mean")
    )
)

summary["tasa_real"] *= 100

summary["diferencia"] = (
    summary["tasa_real"]
    - summary["probabilidad_media"] * 100
)

print(summary.to_string())

In [ ]:
print("=" * 70)
print("DISTRIBUCIÓN DEL RETORNO FUTURO - OOS")
print("=" * 70)

print(oos["future_return"].describe())

print("\nPercentiles:")

print(
    oos["future_return"]
    .quantile([
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ])
)

print("\nCantidad de movimientos:")

print(
    pd.Series({
        "< -0.2%": (oos["future_return"] < -0.002).sum(),
        "-0.2% a +0.2%": (
            (oos["future_return"] >= -0.002) &
            (oos["future_return"] < 0.002)
        ).sum(),
        ">= +0.2%": (oos["future_return"] >= 0.002).sum()
    })
)

In [ ]:
print("=" * 70)
print("DISTRIBUCIÓN DEL TARGET POR PERÍODO")
print("=" * 70)

for nombre, data in [
    ("TRAIN", train),
    ("VALIDATION", validation),
    ("OOS", oos)
]:
    print(f"\n{nombre}")
    print("-" * 40)

    print("Cantidad:")
    print(data["target"].value_counts().sort_index())

    print("\nPorcentaje:")
    print(
        (data["target"]
         .value_counts(normalize=True)
         .sort_index() * 100)
         .round(2)
    )

In [ ]:
print("=" * 70)
print("TASA DE POSITIVOS SEGÚN THRESHOLD")
print("=" * 70)

thresholds = [
    0.001,
    0.0015,
    0.002,
    0.0025,
    0.003,
    0.004
]

for threshold in thresholds:

    print(f"\nThreshold >= {threshold*100:.2f}%")

    for nombre, data in [
        ("TRAIN", train),
        ("VALIDATION", validation),
        ("OOS", oos)
    ]:

        porcentaje = (
            data["future_return"] >= threshold
        ).mean() * 100

        print(f"{nombre:12s}: {porcentaje:.2f}%")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

print("=" * 70)
print("OPTIMIZACIÓN DEL THRESHOLD - VALIDATION")
print("=" * 70)

# ----------------------------------------------------------
# PR-AUC
# ----------------------------------------------------------

pr_auc = average_precision_score(
    y_validation,
    validation_proba
)

print(f"\nPR-AUC Validation: {pr_auc:.4f}")


# ----------------------------------------------------------
# PROBAR DISTINTOS THRESHOLDS
# ----------------------------------------------------------

resultados_threshold = []

for threshold in [x / 100 for x in range(30, 71, 2)]:

    validation_pred = (
        validation_proba >= threshold
    ).astype(int)

    resultados_threshold.append({
        "threshold": threshold,

        "accuracy": accuracy_score(
            y_validation,
            validation_pred
        ),

        "balanced_accuracy": balanced_accuracy_score(
            y_validation,
            validation_pred
        ),

        "precision": precision_score(
            y_validation,
            validation_pred,
            zero_division=0
        ),

        "recall": recall_score(
            y_validation,
            validation_pred,
            zero_division=0
        ),

        "f1": f1_score(
            y_validation,
            validation_pred,
            zero_division=0
        )
    })


threshold_df = pd.DataFrame(resultados_threshold)


# ----------------------------------------------------------
# MOSTRAR TABLA
# ----------------------------------------------------------

print("\nResultados:")
print(
    threshold_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ----------------------------------------------------------
# MEJOR THRESHOLD SEGÚN F1
# ----------------------------------------------------------

mejor_f1 = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

print("\n" + "=" * 70)
print("MEJOR THRESHOLD SEGÚN F1")
print("=" * 70)

print(
    f"Threshold:          {mejor_f1['threshold']:.2f}"
)

print(
    f"Accuracy:           {mejor_f1['accuracy']:.4f}"
)

print(
    f"Balanced Accuracy:  {mejor_f1['balanced_accuracy']:.4f}"
)

print(
    f"Precision:          {mejor_f1['precision']:.4f}"
)

print(
    f"Recall:             {mejor_f1['recall']:.4f}"
)

print(
    f"F1:                 {mejor_f1['f1']:.4f}"
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

BEST_THRESHOLD = 0.40

# Predicciones OOS usando el threshold elegido en VALIDATION
oos_pred_040 = (
    oos_proba >= BEST_THRESHOLD
).astype(int)

print("=" * 70)
print("EVALUACIÓN OOS CON THRESHOLD ELEGIDO EN VALIDATION")
print("=" * 70)

print(f"\nThreshold utilizado: {BEST_THRESHOLD:.2f}")

print(
    f"Accuracy:           "
    f"{accuracy_score(y_oos, oos_pred_040):.4f}"
)

print(
    f"Balanced Accuracy:  "
    f"{balanced_accuracy_score(y_oos, oos_pred_040):.4f}"
)

print(
    f"Precision:          "
    f"{precision_score(y_oos, oos_pred_040, zero_division=0):.4f}"
)

print(
    f"Recall:             "
    f"{recall_score(y_oos, oos_pred_040, zero_division=0):.4f}"
)

print(
    f"F1:                 "
    f"{f1_score(y_oos, oos_pred_040, zero_division=0):.4f}"
)

print("\nMATRIZ DE CONFUSIÓN:")
print(
    pd.DataFrame(
        confusion_matrix(y_oos, oos_pred_040),
        index=["Real 0", "Real 1"],
        columns=["Pred 0", "Pred 1"]
    )
)

print("\nDISTRIBUCIÓN DE PREDICCIONES:")
print(
    pd.Series(oos_pred_040)
    .value_counts()
    .sort_index()
)

print("\nPORCENTAJES:")
print(
    pd.Series(oos_pred_040)
    .value_counts(normalize=True)
    .sort_index()
)

In [ ]:
print("=" * 70)
print("MAPEO DE PROBABILIDAD ORIGINAL → CALIBRADA")
print("=" * 70)

puntos = np.array([
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80
])

calibradas = calibrador.predict(puntos)

for original, calibrada in zip(puntos, calibradas):
    print(
        f"{original:.2f}  →  {calibrada:.4f}"
    )

In [ ]:
print("=" * 70)
print("EVALUACIÓN DE CALIBRACIÓN EN OOS")
print("=" * 70)

# Aplicar el calibrador aprendido en Validation al OOS
oos_proba_cal = calibrador.predict(oos_proba)

# Brier Score OOS
from sklearn.metrics import brier_score_loss

brier_oos_original = brier_score_loss(y_oos, oos_proba)
brier_oos_calibrado = brier_score_loss(y_oos, oos_proba_cal)

print(f"Brier OOS Original:   {brier_oos_original:.6f}")
print(f"Brier OOS Calibrado:  {brier_oos_calibrado:.6f}")

# Estadísticas
print("\nProbabilidades OOS:")
print(f"Media original:       {oos_proba.mean():.6f}")
print(f"Media calibrada:      {oos_proba_cal.mean():.6f}")
print(f"Mediana original:     {np.median(oos_proba):.6f}")
print(f"Mediana calibrada:    {np.median(oos_proba_cal):.6f}")

In [ ]:
import pandas as pd

df_check = pd.read_csv(
    'SOLUSDT_5m_futures.csv',
    nrows=5
)

print("Columnas del CSV:")
for col in df_check.columns:
    print(repr(col))

In [ ]:
# ================================================================
# SELECCIÓN Y CONGELADO DEL UMBRAL SOBRE VALIDATION CALIBRADA
# ================================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import numpy as np

print("=" * 70)
print("SELECCIÓN DE UMBRAL - VALIDATION CALIBRADA")
print("=" * 70)

# Umbrales candidatos
thresholds = np.arange(0.05, 0.81, 0.01)

resultados_threshold = []

for threshold in thresholds:

    pred = (validation_proba_cal >= threshold).astype(int)

    resultados_threshold.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_validation, pred),
        "balanced_accuracy": balanced_accuracy_score(y_validation, pred),
        "precision": precision_score(y_validation, pred, zero_division=0),
        "recall": recall_score(y_validation, pred, zero_division=0),
        "f1": f1_score(y_validation, pred, zero_division=0)
    })

# Convertir a DataFrame
df_thresholds = pd.DataFrame(resultados_threshold)

# Elegimos el umbral que maximiza F1
mejor_fila = df_thresholds.loc[df_thresholds["f1"].idxmax()]

THRESHOLD_CALIBRATED = float(mejor_fila["threshold"])

print("\nUMBRAL SELECCIONADO Y CONGELADO")
print(f"Threshold:           {THRESHOLD_CALIBRATED:.2f}")
print(f"Accuracy:            {mejor_fila['accuracy']:.4f}")
print(f"Balanced Accuracy:   {mejor_fila['balanced_accuracy']:.4f}")
print(f"Precision:            {mejor_fila['precision']:.4f}")
print(f"Recall:               {mejor_fila['recall']:.4f}")
print(f"F1:                   {mejor_fila['f1']:.4f}")

print("\nIMPORTANTE:")
print("Este threshold fue elegido SOLO con Validation.")
print("NO se utilizó OOS para elegirlo.")
print("El threshold queda congelado para la evaluación OOS.")

In [ ]:
# ================================================================
# EVALUACIÓN FINAL OOS - THRESHOLD CONGELADO
# ================================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 70)
print("EVALUACIÓN FINAL OOS - THRESHOLD CONGELADO")
print("=" * 70)

# Aplicar el threshold elegido exclusivamente en Validation
oos_pred_final = (oos_proba_cal >= THRESHOLD_CALIBRATED).astype(int)

# Métricas
accuracy_oos = accuracy_score(y_oos, oos_pred_final)
balanced_acc_oos = balanced_accuracy_score(y_oos, oos_pred_final)
precision_oos = precision_score(y_oos, oos_pred_final, zero_division=0)
recall_oos = recall_score(y_oos, oos_pred_final, zero_division=0)
f1_oos = f1_score(y_oos, oos_pred_final, zero_division=0)

cm_oos = confusion_matrix(y_oos, oos_pred_final)

print(f"\nThreshold calibrado: {THRESHOLD_CALIBRATED:.2f}")

print(f"\nAccuracy:            {accuracy_oos:.4f}")
print(f"Balanced Accuracy:   {balanced_acc_oos:.4f}")
print(f"Precision:            {precision_oos:.4f}")
print(f"Recall:               {recall_oos:.4f}")
print(f"F1:                   {f1_oos:.4f}")

print("\nMatriz de confusión:")
print("              Pred 0    Pred 1")
print(f"Real 0       {cm_oos[0,0]:8d}  {cm_oos[0,1]:8d}")
print(f"Real 1       {cm_oos[1,0]:8d}  {cm_oos[1,1]:8d}")

print("\nDistribución de predicciones:")
print(f"Predicción 0: {(oos_pred_final == 0).sum():,} "
      f"({(oos_pred_final == 0).mean():.2%})")
print(f"Predicción 1: {(oos_pred_final == 1).sum():,} "
      f"({(oos_pred_final == 1).mean():.2%})")

In [ ]:
# ================================================================
# BACKTEST OOS - MODELO CON THRESHOLD CONGELADO
# ================================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("BACKTEST OOS")
print("=" * 70)

# ------------------------------------------------
# PARÁMETROS DEL BACKTEST
# ------------------------------------------------

HORIZON_BT = 3

# Costos simulados por operación completa.
# Se pueden modificar posteriormente para hacer sensibilidad.
FEE_ENTRY = 0.001       # 0.10%
FEE_EXIT = 0.001        # 0.10%
SLIPPAGE_ENTRY = 0.0005 # 0.05%
SLIPPAGE_EXIT = 0.0005  # 0.05%

COST_TOTAL = (
    FEE_ENTRY +
    FEE_EXIT +
    SLIPPAGE_ENTRY +
    SLIPPAGE_EXIT
)

print(f"Horizonte: {HORIZON_BT} velas")
print(f"Coste total simulado: {COST_TOTAL:.3%}")

# ------------------------------------------------
# CONSTRUIR DATAFRAME OOS
# ------------------------------------------------

bt = df.loc[X_oos.index].copy()

bt["prediction"] = oos_pred_final
bt["probability_calibrated"] = oos_proba_cal

# ------------------------------------------------
# BACKTEST CRONOLÓGICO
# ------------------------------------------------

trades = []

i = 0

while i < len(bt) - HORIZON_BT - 1:

    # No hay señal
    if bt["prediction"].iloc[i] != 1:
        i += 1
        continue

    # --------------------------------------------
    # Entrada:
    # apertura de la vela siguiente
    # --------------------------------------------

    entry_idx = i + 1

    # Salida:
    # cierre de la vela HORIZON_BT después de la entrada
    exit_idx = entry_idx + HORIZON_BT - 1

    if exit_idx >= len(bt):
        break

    entry_price = bt["open"].iloc[entry_idx]
    exit_price = bt["close"].iloc[exit_idx]

    entry_time = bt["timestamp"].iloc[entry_idx]
    exit_time = bt["timestamp"].iloc[exit_idx]

    # --------------------------------------------
    # Retorno bruto
    # --------------------------------------------

    gross_return = exit_price / entry_price - 1

    # --------------------------------------------
    # Costos
    # --------------------------------------------

    net_return = gross_return - COST_TOTAL

    trades.append({
        "entry_time": entry_time,
        "exit_time": exit_time,
        "entry_price": entry_price,
        "exit_price": exit_price,
        "probability": bt["probability_calibrated"].iloc[i],
        "gross_return": gross_return,
        "net_return": net_return
    })

    # No permitir operaciones simultáneas
    i = exit_idx + 1


# ------------------------------------------------
# RESULTADOS
# ------------------------------------------------

trades_df = pd.DataFrame(trades)

if len(trades_df) == 0:

    print("\nNo se generaron operaciones.")

else:

    trades_df["equity"] = (1 + trades_df["net_return"]).cumprod()

    total_return = trades_df["equity"].iloc[-1] - 1

    win_rate = (trades_df["net_return"] > 0).mean()

    avg_trade = trades_df["net_return"].mean()

    median_trade = trades_df["net_return"].median()

    best_trade = trades_df["net_return"].max()

    worst_trade = trades_df["net_return"].min()

    # --------------------------------------------
    # Drawdown
    # --------------------------------------------

    equity_curve = trades_df["equity"]

    running_max = equity_curve.cummax()

    drawdown = equity_curve / running_max - 1

    max_drawdown = drawdown.min()

    # --------------------------------------------
    # Profit Factor
    # --------------------------------------------

    ganancias = trades_df.loc[
        trades_df["net_return"] > 0,
        "net_return"
    ].sum()

    perdidas = abs(
        trades_df.loc[
            trades_df["net_return"] < 0,
            "net_return"
        ].sum()
    )

    if perdidas > 0:
        profit_factor = ganancias / perdidas
    else:
        profit_factor = np.inf

    # --------------------------------------------
    # RESULTADOS
    # --------------------------------------------

    print("\nRESULTADOS")
    print("-" * 70)

    print(f"Operaciones:          {len(trades_df):,}")
    print(f"Win Rate:             {win_rate:.2%}")
    print(f"Retorno medio/trade:  {avg_trade:.4%}")
    print(f"Mediana/trade:        {median_trade:.4%}")
    print(f"Mejor trade:          {best_trade:.4%}")
    print(f"Peor trade:           {worst_trade:.4%}")
    print(f"Retorno acumulado:    {total_return:.2%}")
    print(f"Max Drawdown:         {max_drawdown:.2%}")
    print(f"Profit Factor:        {profit_factor:.3f}")

    print("\nPrimeras operaciones:")
    print(
        trades_df[
            [
                "entry_time",
                "exit_time",
                "entry_price",
                "exit_price",
                "probability",
                "gross_return",
                "net_return"
            ]
        ].head(10).to_string(index=False)
    )



    # ================================================================
# AUDITORÍA V1 - ALINEACIÓN TARGET VS BACKTEST
# ================================================================

print("=" * 70)
print("AUDITORÍA V1 - TARGET VS BACKTEST")
print("=" * 70)

# ------------------------------------------------
# Reconstruir exactamente el retorno que usa el backtest
# para cada señal del OOS
# ------------------------------------------------

audit = df.loc[X_oos.index].copy()

audit["prediction"] = oos_pred_final
audit["probability_calibrated"] = oos_proba_cal

# Entrada: apertura de la vela siguiente
audit["bt_entry_price"] = audit["open"].shift(-1)

# Salida: cierre de la tercera vela desde la entrada
# i -> entrada i+1 -> salida i+3
audit["bt_exit_price"] = audit["close"].shift(-HORIZON_BT)

audit["bt_gross_return"] = (
    audit["bt_exit_price"] /
    audit["bt_entry_price"]
    - 1
)

# ------------------------------------------------
# Comparar con el future_return usado por el target
# ------------------------------------------------

audit["target_from_bt"] = (
    audit["bt_gross_return"] >= TARGET_THRESHOLD
).astype(int)

valid = audit["bt_gross_return"].notna()

same_target = (
    audit.loc[valid, "target"].values ==
    audit.loc[valid, "target_from_bt"].values
)

print("\nALINEACIÓN")
print("-" * 70)

print(f"Observaciones comparadas: {valid.sum():,}")
print(f"Coinciden target/backtest: {same_target.sum():,}")
print(f"No coinciden:             {(~same_target).sum():,}")
print(f"Coincidencia:             {same_target.mean():.4%}")

# ------------------------------------------------
# Distribución de retornos de las señales
# ------------------------------------------------

signals = audit[
    (audit["prediction"] == 1) &
    (audit["bt_gross_return"].notna())
].copy()

print("\nRETORNO BRUTO DE LAS SEÑALES")
print("-" * 70)

print(f"Señales:                  {len(signals):,}")
print(f"Media:                    {signals['bt_gross_return'].mean():.4%}")
print(f"Mediana:                  {signals['bt_gross_return'].median():.4%}")
print(f"Percentil 25:             {signals['bt_gross_return'].quantile(.25):.4%}")
print(f"Percentil 75:             {signals['bt_gross_return'].quantile(.75):.4%}")
print(f"Percentil 90:             {signals['bt_gross_return'].quantile(.90):.4%}")
print(f"Percentil 95:             {signals['bt_gross_return'].quantile(.95):.4%}")

print("\nPOR UMBRAL DE RETORNO BRUTO")
print("-" * 70)

for threshold in [0.001, 0.002, 0.003, 0.004, 0.005]:

    rate = (
        signals["bt_gross_return"] >= threshold
    ).mean()

    print(
        f">= {threshold:.2%}: "
        f"{rate:.2%} "
        f"({(signals['bt_gross_return'] >= threshold).sum():,} operaciones)"
    )